# Keratoconus Detection: Multimodal Fusion Framework
**Research Scholar: Rocky Dewan** | **Dataset:** Zenodo ID 17127265

---


## Notebook Structure

**Definition Cells (01-12):** All classes and functions defined -- no training yet.

**Execution Cells (13-23):** Each cell runs immediately and produces visible output.

| Cell | Content |
|------|--------|
| 13 | Data loading + clinical preprocessing |
| 14 | Per-channel registration + normalizer + dataloaders |
| 15 | **APEX model training** (ViT-B/16 + TabTransformer + CrossAttn) |
| 16 | **SOTA baselines** (DenseNet-121/201, ResNet-50, EfficientNet, unimodal) |
| 17 | **Fusion strategy comparison** (Early/Late/Intermediate/Hybrid) |
| 18 | **Ablation studies** (components + preprocessing + augmentation) |
| 19 | **Modality dropout** robustness + **data efficiency** (stratified) |
| 20 | **Grad-CAM** heatmaps + tabular feature importance |
| 21 | **Metrics** + **bootstrap 95% CIs** (N=2000) + **NRI table** (all pairs) |
| 22 | **Visualization suite** (Radar, Ablation bars, Calibration, CI chart, LR) |


## 🛠️ Audit Fix Summary — Applied Corrections (22 May 2026)

| Fix | Severity | Location | Change |
|-----|----------|----------|--------|
| **§1 Patient-level split** | CRITICAL | `DataAuditor._split()` (Cell 2) | Row-level `train_test_split` → patient-ID-level `train_test_split`; all eye records (OD/OS) stay in the same partition. Zero bilateral leakage guaranteed by assertion. |
| **§2 timm assertion** | HIGH | `PretrainedViTEncoder` (Cell 5) | Silent `try/except` fallback to `_ScratchViT` replaced by hard `assert _TIMM_OK`. `_ScratchViT` class removed. Pretrained weights required. |
| **§3a ViT projection dropout** | HIGH | `PretrainedViTEncoder` (Cell 5) | Dropout in projection head: `0.1 → 0.4` |
| **§3b TabTransformer depth** | HIGH | `TabTransformerEncoder` (Cell 5) | Depth `4 → 2`; dropout `0.1 → 0.3` |
| **§3c CrossAttention dropout** | HIGH | `CrossAttentionBlock` (Cell 6) | Dropout `0.1 → 0.4` |
| **§3d APEX classifier head** | HIGH | `AttentionIntermediateFusionNet` (Cell 7) | Added second Dropout+Linear stage (128-D); default dropout `0.1 → 0.4` |
| **§3e Ablation/baseline dropout** | MEDIUM | Cells 5, 8 | `ClinicalOnlyTabTransformer`, `AblationNoCrossAttn`, `AblationNoTabTransformer` dropout `0.1 → 0.4` |
| **§3f FocalLoss label smoothing** | MEDIUM | `FocalLoss` (Cell 11) | `label_smoothing=0.1` added to training objective |
| **§5 NUM_EPOCHS** | LOW | Cell 1 global constants | `50 → 100` (per Implementation Plan §9) |
| **§4 5-Fold Patient CV** | STRATEGY | Cell 38 | Full `run_stratified_cv()` with `StratifiedGroupKFold` on `patient_code`; per-fold `ClinicalPreprocessor` re-fit; bootstrap 95% CIs (N=2000) for AUC, F1, MCC |

> **ClinicalPreprocessor note:** `KNNImputer`, `IQROutlierCapper`, and `MinMaxScaler` are already correctly called via `.fit_transform()` on `train_df` and `.transform()` on `val_df`/`test_df` in Cell 14. With the patient-level split fix, these preprocessors are now truly leak-free.


In [ ]:

# SECTION 0: GLOBAL SETUP, IMPORTS & REPRODUCIBILITY


import os, sys, random, warnings, copy, json, time, math
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd

import matplotlib
try:
    shell = get_ipython().__class__.__name__   
    if shell in ('ZMQInteractiveShell', 'TerminalInteractiveShell'):
        matplotlib.use('module://matplotlib_inline.backend_inline')
    else:
        matplotlib.use('Agg')
except NameError:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from scipy import stats
from scipy.ndimage import gaussian_filter, map_coordinates

import matplotlib.gridspec as gridspec

from PIL import Image
from scipy.interpolate import make_interp_spline
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import StratifiedKFold
import os, math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
from torchvision import models

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score, recall_score, precision_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    roc_curve, precision_recall_curve,
)

warnings.filterwarnings('ignore')

IMAGE_BASE_DIR = '/kaggle/input/datasets/rockydewan/oct-and-ca/ORBSCAN_Dataset/ORBSCAN_Dataset'
CSV_PATH       = '/kaggle/input/datasets/rockydewan/oct-and-ca/clinical_data_and_labels.csv'
OUTPUT_DIR     = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE          = 224
NUM_CHANNELS      = 4
NUM_CLASSES       = 2
MAP_TYPES         = ['Anterior', 'Axial', 'Pachymetry', 'Posterior']
EYES              = ['OD', 'OS']
CLASS_NAMES       = ['Normal', 'Keratoconus']

NUMERICAL_FEATURES = [
    'age_years', 'astig_value_d', 'astig_axis_deg', 'kmax_value_d',
    'kmax_axis_deg', 'pachy_central_um', 'pachy_thinnest_um',
    'pachy_thinnest_x', 'pachy_thinnest_y',
    'asphericity_anterior', 'asphericity_posterior',
]
CATEGORICAL_FEATURES = ['gender']

BATCH_SIZE   = 32
NUM_EPOCHS   = 100  
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
SEED         = 42

N_BOOTSTRAP  = 2000

def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED']       = str(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("  SECTION 0: ENVIRONMENT VERIFIED")
print(f"  Device      : {DEVICE}")
print(f"  PyTorch     : {torch.__version__}")
if torch.cuda.is_available():
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  N_BOOTSTRAP : {N_BOOTSTRAP}  <- 2000 (PhD thesis standard)")


In [ ]:

# SECTION 1: DATASET ANALYSIS & PREPARATION


class DataAuditor:
    def __init__(self, image_base_dir=IMAGE_BASE_DIR, csv_path=CSV_PATH,
                 missing_strategy='exclude', val_size=0.15, test_size=0.15, seed=SEED):
        self.image_base_dir = Path(image_base_dir); self.csv_path = Path(csv_path)
        self.missing_strategy = missing_strategy
        self.val_size = val_size; self.test_size = test_size; self.seed = seed

    def _scan_images(self) -> pd.DataFrame:
        if not self.image_base_dir.exists():
            raise FileNotFoundError(f"IMAGE_BASE_DIR not found: {self.image_base_dir}")
        records = []
        pdirs = sorted([d for d in self.image_base_dir.iterdir() if d.is_dir()])
        print(f"  Scanning {len(pdirs)} patient dirs...")
        for i, pdir in enumerate(pdirs):
            pc = pdir.name
            if (i+1) % 100 == 0: print(f"    {i+1}/{len(pdirs)}", end='\r')
            for ed in sorted(pdir.iterdir()):
                if not ed.is_dir(): continue
                eye = ed.name.strip().upper()
                if eye not in EYES: continue
                paths = {}; complete = True
                for mt in MAP_TYPES:
                    exp = ed / f"{pc}_{eye}_{mt}.png"
                    if exp.exists():
                        try: Image.open(exp).verify(); paths[mt] = str(exp)
                        except: paths[mt] = None; complete = False
                    else:
                        fb = next((f for f in ed.iterdir()
                                   if f.name.lower() == f"{pc}_{eye}_{mt}.png".lower()), None)
                        paths[mt] = str(fb) if fb else None
                        if not fb: complete = False
                records.append({'patient_code': pc, 'eye': eye, **paths, 'complete': complete})
        print()
        df = pd.DataFrame(records)
        print(f"  patient-eye pairs: {len(df)} | complete: {df['complete'].sum()}")
        return df

    def _merge_clinical(self, image_df):
        cdf = pd.read_csv(self.csv_path)
        print(f"  CSV shape: {cdf.shape}")
        cdf.columns = [str(c).strip().lower().replace(' ','_').replace('-','_') for c in cdf.columns]
        for cand in ['patient_code','patientcode','patient_id','id','subject_id','code']:
            if cand in cdf.columns: cdf.rename(columns={cand:'patient_code'}, inplace=True); break
        for cand in ['eye','side','laterality']:
            if cand in cdf.columns: cdf.rename(columns={cand:'eye'}, inplace=True); break
        cdf['eye'] = cdf['eye'].astype(str).str.strip().str.upper().replace({'L':'OS','R':'OD','LEFT':'OS','RIGHT':'OD'})
        cdf['patient_code'] = cdf['patient_code'].astype(str).str.strip()
        image_df = image_df.copy(); image_df['patient_code'] = image_df['patient_code'].astype(str).str.strip()
        merged = pd.merge(image_df, cdf, on=['patient_code','eye'], how='inner')
        print(f"  Merged rows: {len(merged)}"); return merged

    def _handle_missing(self, df):
        if self.missing_strategy == 'exclude':
            b = len(df); df = df[df['complete']].reset_index(drop=True)
            print(f"  [exclude] {b} -> {len(df)}")
        elif self.missing_strategy == 'missing_aware':
            for mt in MAP_TYPES: df[f'{mt}_present'] = df[mt].notna().astype(int)
        return df

    def _split(self, df):

        for cand in ['label','diagnosis','class','keratoconus','target','y']:
            if cand in df.columns:
                df = df.rename(columns={cand: 'label'}) if cand != 'label' else df
                break
        if 'label' not in df.columns:
            for c in df.columns:
                if df[c].nunique() == 2:
                    df = df.rename(columns={c: 'label'}); break
        if df['label'].dtype == object:
            mp = {}
            for v in df['label'].astype(str).str.lower().unique():
                if   v in ('0','normal','no','negative','healthy','n'): mp[v] = 0
                elif v in ('1','keratoconus','kc','yes','positive','disease','k'): mp[v] = 1
            df['label'] = df['label'].astype(str).str.lower().map(mp).fillna(0).astype(int)
        else:
            df['label'] = df['label'].astype(int)


        patient_labels = (
            df.groupby('patient_code')['label']
              .agg(lambda x: int(x.mode().iloc[0]))
              .reset_index()
              .rename(columns={'label': 'pt_label'})
        )
        pts   = patient_labels['patient_code'].values
        plbls = patient_labels['pt_label'].values
        print(f"  [Split] Unique patients: {len(pts)}  | KC patients: {plbls.sum()}")


        from sklearn.model_selection import train_test_split as _tts
        tv_pts, test_pts, tv_lbl, _ = _tts(
            pts, plbls,
            test_size=self.test_size,
            stratify=plbls,
            random_state=self.seed
        )
        train_pts, val_pts, _, _ = _tts(
            tv_pts, tv_lbl,
            test_size=self.val_size / (1.0 - self.test_size),
            stratify=tv_lbl,
            random_state=self.seed
        )


        train = df[df['patient_code'].isin(set(train_pts))].reset_index(drop=True)
        val   = df[df['patient_code'].isin(set(val_pts))  ].reset_index(drop=True)
        test  = df[df['patient_code'].isin(set(test_pts)) ].reset_index(drop=True)

        for nm, sdf in [('Train(70%)', train), ('Val(15%)', val), ('Test(15%)', test)]:
            vc   = sdf['label'].value_counts().sort_index()
            npts = sdf['patient_code'].nunique()
            print(f"  {nm}: {len(sdf):5d} eye-rows | {npts:4d} patients "
                  f"| N={vc.get(0,0)} KC={vc.get(1,0)}")

        assert not (set(train_pts) & set(val_pts))  and                not (set(train_pts) & set(test_pts)),             "Patient overlap detected between partitions!"
        print("  [Split] Patient-level isolation verified ✓ — zero cross-partition leakage.")

        return train, val, test
    def run(self):
        imgs   = self._scan_images()
        merged = self._merge_clinical(imgs)
        merged = self._handle_missing(merged)
        train, val, test = self._split(merged)
        return merged, train, val, test

print("Section 1: DataAuditor defined.")


In [ ]:

# SECTION 2: PREPROCESSING (MODALITY-SPECIFIC)


# 2.1  MEDICAL IMAGING 

class IsotropicResampler:
    """Resize corneal map to IMG_SIZE x IMG_SIZE (2D isotropic pixel grid)."""
    def __init__(self, target_size=IMG_SIZE): self.target_size = target_size
    def resample(self, arr):
        pil = Image.fromarray((arr*255).astype(np.uint8), mode='L')
        return np.array(pil.resize((self.target_size,self.target_size), Image.BILINEAR), dtype=np.float32)/255.0


class AffineRegistrar:
    """
    Phase-correlation rigid registration to a per-channel mean template.

    """
    def __init__(self):
        self.template = None

    def fit(self, images: np.ndarray) -> 'AffineRegistrar':
        """images: (N, H, W) float32 -- all from the SAME channel."""
        self.template = images.mean(axis=0); return self

    def register_one(self, img: np.ndarray) -> np.ndarray:
        if self.template is None: return img
        from scipy.ndimage import shift as nd_shift
        F_i  = np.fft.fft2(img); F_t = np.fft.fft2(self.template)
        cross = F_t * np.conj(F_i); R = cross / (np.abs(cross)+1e-8)
        r     = np.fft.ifft2(R).real
        idx   = np.unravel_index(np.argmax(r), r.shape)
        dy = idx[0] if idx[0]<img.shape[0]//2 else idx[0]-img.shape[0]
        dx = idx[1] if idx[1]<img.shape[1]//2 else idx[1]-img.shape[1]
        return nd_shift(img, shift=(-dy,-dx), mode='reflect').astype(np.float32)


class ElasticTransform:
    def __init__(self, alpha=50.0, sigma=5.0, p=0.3):
        self.alpha=alpha; self.sigma=sigma; self.p=p
    def __call__(self, img):
        if random.random()>self.p: return img
        H,W=img.shape
        dx=gaussian_filter((np.random.rand(H,W)*2-1),self.sigma)*self.alpha
        dy=gaussian_filter((np.random.rand(H,W)*2-1),self.sigma)*self.alpha
        x,y=np.meshgrid(np.arange(W),np.arange(H))
        xn=np.clip(x+dx,0,W-1).astype(np.float32)
        yn=np.clip(y+dy,0,H-1).astype(np.float32)
        return map_coordinates(img,[yn.ravel(),xn.ravel()],order=1).reshape(H,W).astype(np.float32)


def add_gaussian_noise(img, std=0.02):
    return np.clip(img+np.random.normal(0,std,img.shape),0,1).astype(np.float32)

def get_train_transforms():
    return T.Compose([
        T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomAffine(degrees=15,translate=(0.1,0.1),scale=(0.9,1.1),shear=5),
        T.RandomApply([T.GaussianBlur(kernel_size=3,sigma=(0.1,2.0))],p=0.3),
        T.ToTensor()])

def get_val_transforms():
    return T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor()])


# 2.2  CLINICAL DATA 

class IQROutlierCapper:
    def __init__(self, factor=1.5): self.factor=factor; self.bounds={}
    def fit(self, df, cols):
        for c in cols:
            if c not in df.columns: continue
            q1,q3=df[c].quantile([0.25,0.75]); iqr=q3-q1
            self.bounds[c]=(q1-self.factor*iqr, q3+self.factor*iqr)
        return self
    def transform(self, df):
        df=df.copy()
        for c,(lo,hi) in self.bounds.items():
            if c in df.columns: df[c]=df[c].clip(lower=lo,upper=hi)
        return df


class ClinicalAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(input_dim,32),nn.BatchNorm1d(32),nn.GELU(),nn.Linear(32,latent_dim))
        self.decoder=nn.Sequential(nn.Linear(latent_dim,32),nn.GELU(),nn.Linear(32,input_dim))
        self.latent_dim=latent_dim
    def forward(self,x): z=self.encoder(x); return z,self.decoder(z)
    def fit(self,X,n_epochs=100,lr=1e-3,device=DEVICE):
        self.to(device).train(); opt=optim.Adam(self.parameters(),lr=lr)
        ds=DataLoader(torch.utils.data.TensorDataset(torch.tensor(X,dtype=torch.float32,device=device)),batch_size=32,shuffle=True)
        for _ in range(n_epochs):
            for (b,) in ds: opt.zero_grad(); _,r=self(b); F.mse_loss(r,b).backward(); opt.step()
        self.eval(); return self
    @torch.no_grad()
    def encode(self,X,device=DEVICE):
        self.to(device).eval()
        return self.encoder(torch.tensor(X,dtype=torch.float32,device=device)).cpu().numpy()


class ClinicalPreprocessor:
    """IQR capping -> imputation -> scaling -> encoding -> (optional DR).
    ."""
    def __init__(self, imputer='knn', scaler='minmax', cat_encoding='onehot',
                 dim_reduction='none', pca_components=8, ae_latent_dim=8):
        self.dim_reduction=dim_reduction; self.ae_latent_dim=ae_latent_dim
        self.capper=IQROutlierCapper(); self._fitted=False; self._dim=0
        imp={'median':SimpleImputer(strategy='median'),
             'knn':KNNImputer(n_neighbors=5),
             'iterative':IterativeImputer(max_iter=10,random_state=SEED)}[imputer]
        self.num_pipe=Pipeline([('imp',imp),('sc',MinMaxScaler() if scaler=='minmax' else StandardScaler())])
        self.cat_imp=SimpleImputer(strategy='most_frequent'); self.cat_enc_type=cat_encoding
        self.cat_enc=(OneHotEncoder(handle_unknown='ignore',sparse_output=False)
                      if cat_encoding=='onehot' else LabelEncoder())
        self.pca=PCA(n_components=pca_components,random_state=SEED) if dim_reduction=='pca' else None
        self.ae=None

    def _get_cols(self,df): return ([c for c in NUMERICAL_FEATURES if c in df.columns],[c for c in CATEGORICAL_FEATURES if c in df.columns])

    def _encode_cat(self,df,cat_cols,fit=False):
        if not cat_cols: return np.empty((len(df),0),dtype=np.float32)
        arr=df[cat_cols].astype(str).values
        arr=self.cat_imp.fit_transform(arr) if fit else self.cat_imp.transform(arr)
        if self.cat_enc_type=='onehot':
            enc=self.cat_enc.fit_transform(arr) if fit else self.cat_enc.transform(arr)
        else:
            enc=np.stack([(self.cat_enc.fit_transform(arr[:,i]) if fit else self.cat_enc.transform(arr[:,i])) for i in range(arr.shape[1])],axis=-1)
        return enc.astype(np.float32)

    def fit_transform(self,df):
        nc,cc=self._get_cols(df); cap=self.capper.fit(df,nc).transform(df)
        na=self.num_pipe.fit_transform(cap[nc]).astype(np.float32) if nc else np.empty((len(df),0),dtype=np.float32)
        ca=self._encode_cat(cap,cc,fit=True); X=np.concatenate([na,ca],axis=1)
        if self.dim_reduction=='pca': X=self.pca.fit_transform(X).astype(np.float32)
        elif self.dim_reduction=='autoencoder':
            self.ae=ClinicalAutoencoder(X.shape[1],self.ae_latent_dim).fit(X); X=self.ae.encode(X).astype(np.float32)
        self._dim=X.shape[1]; self._fitted=True
        print(f"  [ClinicalPreprocessor] fit_transform: shape={X.shape}"); return X

    def transform(self,df):
        assert self._fitted; nc,cc=self._get_cols(df); cap=self.capper.transform(df)
        na=self.num_pipe.transform(cap[nc]).astype(np.float32) if nc else np.empty((len(df),0),dtype=np.float32)
        ca=self._encode_cat(cap,cc,fit=False); X=np.concatenate([na,ca],axis=1)
        if self.dim_reduction=='pca': X=self.pca.transform(X).astype(np.float32)
        elif self.dim_reduction=='autoencoder' and self.ae: X=self.ae.encode(X).astype(np.float32)
        return X

    @property
    def output_dim(self): return self._dim

print("Section 2: Preprocessing classes defined.")


In [ ]:

# SECTION 3 (DATASET): OrbscanDataset



class OrbscanDataset(Dataset):
    """
    4-channel Orbscan dataset.
    registrar: Dict[channel_name -> AffineRegistrar]
               Pass None to disable registration (ablation).
    """
    def __init__(self, df, clinical_arr, transform=None, augment=False,
                 registrar=None,
                 normalizer_means=None, normalizer_stds=None, missing_strategy='exclude'):
        self.df=df.reset_index(drop=True); self.clinical_arr=clinical_arr
        self.transform=transform; self.augment=augment
        self.registrar=registrar          # Dict[mt -> AffineRegistrar] or None
        self.norm_means=normalizer_means; self.norm_stds=normalizer_stds
        self.missing_strategy=missing_strategy
        self.resampler=IsotropicResampler(IMG_SIZE)
        self.elastic=ElasticTransform(50,5,0.3) if augment else None

    def __len__(self): return len(self.df)

    def _load_channel(self, path):
        if path is None or (isinstance(path,float) and math.isnan(path)):
            return np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.float32)
        arr=np.array(Image.open(path).convert('L'),dtype=np.float32)/255.0
        return self.resampler.resample(arr)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        channels=[self._load_channel(row.get(mt,None)) for mt in MAP_TYPES]


        if self.registrar is not None:
            channels=[
                self.registrar[mt].register_one(c)
                if (mt in self.registrar and self.registrar[mt].template is not None) else c
                for mt,c in zip(MAP_TYPES,channels)]

        if self.augment:
            ch2=[]
            for c in channels:
                if self.elastic: c=self.elastic(c)
                if random.random()<0.3: c=add_gaussian_noise(c)
                ch2.append(c)
            channels=ch2

        if self.transform is not None:
            seed_now=random.randint(0,2**31-1); out=[]
            for c in channels:
                pil=Image.fromarray((c*255).astype(np.uint8),mode='L')
                random.seed(seed_now); torch.manual_seed(seed_now); np.random.seed(seed_now%(2**31))
                out.append(self.transform(pil))
            image_tensor=torch.cat(out,dim=0)
        else:
            image_tensor=torch.from_numpy(np.stack(channels,axis=0))

        if self.norm_means is not None and self.norm_stds is not None:
            for c in range(NUM_CHANNELS):
                image_tensor[c]=(image_tensor[c]-self.norm_means[c])/self.norm_stds[c]

        clin_t=torch.from_numpy(self.clinical_arr[idx])
        if self.missing_strategy=='missing_aware':
            mask=torch.tensor([int(row.get(f'{mt}_present',1)) for mt in MAP_TYPES],dtype=torch.float32)
            return image_tensor, clin_t, torch.tensor(label,dtype=torch.long), mask
        return image_tensor, clin_t, torch.tensor(label,dtype=torch.long)

print("OrbscanDataset: per-channel registrar support.")


In [ ]:

# SECTION 3: FEATURE EXTRACTORS


#  3.1a CNN Encoder ─
class CNNEncoder(nn.Module):
    """
    Pre-trained CNN for 4-channel Orbscan input.

    """
    def __init__(self, backbone='densenet121', embed_dim=512,
                 in_channels=NUM_CHANNELS, pretrained=True, dropout=0.3):
        super().__init__(); self.backbone_name=backbone
        if backbone=='resnet50':
            base=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            base.conv1=self._adapt_conv(base.conv1,in_channels,pretrained)
            feat_dim=base.fc.in_features; base.fc=nn.Identity(); self.encoder=base
        elif backbone=='densenet121':
            base=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
            base.features.conv0=self._adapt_conv(base.features.conv0,in_channels,pretrained)
            base.classifier=nn.Identity(); feat_dim=1024; self.encoder=base
        elif backbone=='densenet201':
            base=models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None)
            base.features.conv0=self._adapt_conv(base.features.conv0,in_channels,pretrained)
            base.classifier=nn.Identity(); feat_dim=1920; self.encoder=base
        else: raise ValueError(f"Unknown backbone: {backbone}")
        self.head=nn.Sequential(nn.Linear(feat_dim,embed_dim),nn.LayerNorm(embed_dim),nn.GELU(),nn.Dropout(dropout))
        self.embed_dim=embed_dim

    @staticmethod
    def _adapt_conv(old_conv, in_channels, pretrained):
        nc=nn.Conv2d(in_channels,old_conv.out_channels,old_conv.kernel_size,old_conv.stride,old_conv.padding,bias=False)
        with torch.no_grad():
            if pretrained and old_conv.weight.shape[1]==3:
                nc.weight[:,:3]=old_conv.weight
                avg=old_conv.weight.mean(dim=1,keepdim=True)
                for i in range(3,in_channels): nc.weight[:,i:i+1]=avg
        return nc

    def forward(self,x):
        if 'densenet' in self.backbone_name:
            f=self.encoder.features(x); f=F.adaptive_avg_pool2d(f,(1,1)).flatten(1)
        else: f=self.encoder(x)
        return self.head(f)


#  Hard assertion — no silent scratch-ViT fallback ──
try:
    import timm
    _TIMM_OK = True
    print(f"  [OK] timm {timm.__version__} loaded successfully.")
except ImportError:
    _TIMM_OK = False

assert _TIMM_OK, (
    "CRITICAL: timm library is missing or improperly installed. "
    "The APEX model CANNOT be trained from scratch on this dataset size (~300-600 samples). "
    "Install via:  pip install timm>=0.9.0  then restart the kernel."
)

class PretrainedViTEncoder(nn.Module):
    """
    Vision Transformer backbone (ViT-B/16) with pretrained ImageNet weights via timm.

    2: Silent scratch-ViT fallback removed. timm is REQUIRED.
    On datasets < 1000 samples a scratch ViT-B (~86M params) catastrophically
    overfits; pretrained weights are non-negotiable.

    4-channel adaptation:
      - Channels 0-2: copy pretrained RGB patch-embed weights directly
      - Channel 3   : initialise as mean of the 3 RGB weight slices
    ViT-B output (768-D) → projection head (Linear + LayerNorm + GELU + Dropout)
    → embed_dim (512-D).

    3: dropout raised to 0.4 in the projection head for regularisation.
    """
    def __init__(self, embed_dim=512, in_channels=NUM_CHANNELS,
                 pretrained=True, dropout=0.4):
        super().__init__()
        self.embed_dim = embed_dim

        # timm guaranteed available (assertion above class definition)
        self.vit = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained,
            num_classes=0, global_pool='token'
        )
        op = self.vit.patch_embed.proj
        np_ = nn.Conv2d(in_channels, op.out_channels, op.kernel_size, op.stride)
        with torch.no_grad():
            np_.weight[:, :3] = op.weight
            np_.weight[:, 3:] = op.weight.mean(dim=1, keepdim=True)
        self.vit.patch_embed.proj = np_

        # 3: projection head with explicit dropout (0.4) for small dataset
        self.proj = nn.Sequential(
            nn.Linear(768, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(p=dropout)
        )

        print(f"  [PretrainedViTEncoder] timm ViT-B/16 loaded | "
              f"pretrained={pretrained} | projection dropout={dropout}")
        print(f"  [PretrainedViTEncoder] 4-ch patch embed: "
              f"ch0-2 from ImageNet RGB, ch3 = mean(RGB channels)")

    def forward(self, x):
        return self.proj(self.vit(x))

# 2: _ScratchViT REMOVED — timm assertion above makes this unreachable.
# A from-scratch ViT with ~86M params catastrophically overfits on <600 samples.
# If needed for an explicit ablation, restore only with a much larger dataset.

class MLPEncoder(nn.Module):
    def __init__(self,input_dim,embed_dim=64,dropout=0.3):
        super().__init__(); self.embed_dim=embed_dim
        self.net=nn.Sequential(
            nn.Linear(input_dim,256),nn.BatchNorm1d(256),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(128,embed_dim),nn.LayerNorm(embed_dim))
    def forward(self,x): return self.net(x)


#  3.2b TabTransformer with CLS-token pooling 
class TabTransformerEncoder(nn.Module):

    # 3: depth reduced 4→2, dropout raised 0.1→0.3 (small-dataset regularisation)
    def __init__(self,input_dim,embed_dim=64,depth=2,n_heads=4,mlp_ratio=2.0,dropout=0.3):
        super().__init__(); self.embed_dim=embed_dim; self.input_dim=input_dim
        self.feature_embed=nn.Linear(1,embed_dim)
        self.pos_embed=nn.Embedding(input_dim,embed_dim)
        self.cls_token=nn.Parameter(torch.zeros(1,1,embed_dim))
        nn.init.trunc_normal_(self.cls_token,std=0.02)
        el=nn.TransformerEncoderLayer(d_model=embed_dim,nhead=n_heads,dim_feedforward=int(embed_dim*mlp_ratio),
                                      dropout=dropout,activation='gelu',batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(el,num_layers=depth)
        self.norm=nn.LayerNorm(embed_dim)

    def forward(self,x):
        B,D=x.shape
        tokens=self.feature_embed(x.unsqueeze(-1))                  # (B, D, E)
        tokens=tokens+self.pos_embed(torch.arange(D,device=x.device)).unsqueeze(0)
        cls=self.cls_token.expand(B,-1,-1)                          # (B, 1, E)
        tokens=torch.cat([cls,tokens],dim=1)                        # (B, D+1, E)
        return self.norm(self.transformer(tokens))[:,0]              # CLS output

print("Section 3: Encoders defined.")
print(f"  PretrainedViTEncoder : timm={_TIMM_OK} (REQUIRED — no fallback)")
print(f"  ViT projection       : dropout=0.4 (3)")
print(f"  TabTransformerEncoder: depth=2, dropout=0.3 (3)")
print("  TabTransformerEncoder: CLS-token pooling")
print("  CNNEncoder           : DenseNet-121 primary")


In [ ]:

# SECTION 4: MULTIMODAL FUSION MODULE



class CrossAttentionBlock(nn.Module):
    """Bidirectional cross-attention: image <-> clinical (plan Section 5)."""
    # 3: dropout raised 0.1 -> 0.4 (small-dataset regularisation)
    def __init__(self,embed_dim=512,n_heads=8,dropout=0.4):
        super().__init__()
        self.attn_i2c=nn.MultiheadAttention(embed_dim,n_heads,dropout=dropout,batch_first=True)
        self.attn_c2i=nn.MultiheadAttention(embed_dim,n_heads,dropout=dropout,batch_first=True)
        self.n1=nn.LayerNorm(embed_dim); self.n2=nn.LayerNorm(embed_dim)
        self.ffn=nn.Sequential(nn.Linear(embed_dim*2,embed_dim*2),nn.GELU(),nn.Dropout(dropout),nn.Linear(embed_dim*2,embed_dim))
        self.n3=nn.LayerNorm(embed_dim)
    def forward(self,img,clin):
        is_=img.unsqueeze(1); cs_=clin.unsqueeze(1)
        ia,_=self.attn_i2c(is_,cs_,cs_); ca,_=self.attn_c2i(cs_,is_,is_)
        io=self.n1(img+ia.squeeze(1)); co=self.n2(clin+ca.squeeze(1))
        return self.n3(self.ffn(torch.cat([io,co],dim=-1)))


class FusionModule(nn.Module):
    """
    Configurable fusion: 'early' | 'late' | 'intermediate' | 'hybrid'.
    """
    def __init__(self,fusion_type,img_encoder,clin_encoder,img_embed_dim,clin_embed_dim,
                 n_classes=NUM_CLASSES,dropout=0.3,n_heads=8,img_raw_dim=0,clin_raw_dim=0,late_combine='average'):
        super().__init__()
        self.ft=fusion_type; self.ie=img_encoder; self.ce=clin_encoder; self.lc=late_combine
        if fusion_type=='early':
            self.clf=nn.Sequential(nn.Linear(img_raw_dim+clin_raw_dim,512),nn.LayerNorm(512),nn.GELU(),nn.Dropout(dropout),
                                   nn.Linear(512,256),nn.GELU(),nn.Dropout(dropout),nn.Linear(256,n_classes))
        elif fusion_type=='late':
            self.ih=nn.Linear(img_embed_dim,n_classes); self.ch=nn.Linear(clin_embed_dim,n_classes)
            if late_combine=='meta': self.meta=nn.Linear(n_classes*2,n_classes)
        elif fusion_type=='intermediate':
            self.fp=nn.Sequential(nn.Linear(img_embed_dim+clin_embed_dim,img_embed_dim),nn.LayerNorm(img_embed_dim),nn.GELU(),nn.Dropout(dropout))
            self.clf=nn.Sequential(nn.Linear(img_embed_dim,128),nn.GELU(),nn.Dropout(dropout),nn.Linear(128,n_classes))
        elif fusion_type=='hybrid':
            self.cp=nn.Linear(clin_embed_dim,img_embed_dim) if clin_embed_dim!=img_embed_dim else nn.Identity()
            self.ep=nn.Sequential(nn.Linear(img_embed_dim*2,img_embed_dim),nn.LayerNorm(img_embed_dim),nn.GELU())
            self.ca=CrossAttentionBlock(img_embed_dim,n_heads)
            self.ih=nn.Linear(img_embed_dim,n_classes); self.ch=nn.Linear(clin_embed_dim,n_classes)
            self.jh=nn.Sequential(nn.Linear(img_embed_dim,128),nn.GELU(),nn.Dropout(dropout),nn.Linear(128,n_classes))
        else: raise ValueError(f"Unknown fusion_type: {fusion_type}")

    def forward(self,img,clin):
        if self.ft=='early': return self.clf(torch.cat([img.flatten(1),clin],dim=-1))
        elif self.ft=='late':
            ie=self.ie(img); ce=self.ce(clin); li=self.ih(ie); lc=self.ch(ce)
            if self.lc=='average': return (li+lc)/2
            elif self.lc=='voting': return F.softmax(li,-1)+F.softmax(lc,-1)
            elif self.lc=='meta': return self.meta(torch.cat([li,lc],-1))
        elif self.ft=='intermediate':
            ie=self.ie(img); ce=self.ce(clin); return self.clf(self.fp(torch.cat([ie,ce],-1)))
        elif self.ft=='hybrid':
            ie=self.ie(img); ce=self.ce(clin); ca=self.cp(ce)
            early=self.ep(torch.cat([ie,ca],-1)); fused=self.ca(early,ca)
            return (self.ih(ie)+self.ch(ce)+self.jh(fused))/3

print("Section 4: FusionModule defined (hybrid dim-mismatch guarded).")


In [ ]:

# SECTION 5: APEX MODEL -- AttentionIntermediateFusionNet


class AttentionIntermediateFusionNet(nn.Module):
    """
      PretrainedViTEncoder  -> 512-D image embedding
      TabTransformerEncoder -> 512-D clinical embedding  (CLS-token pool)
      CrossAttentionBlock   -> 512-D fused representation
      2-layer MLP classifier -> Normal / Keratoconus
    """
    # 3: tab_depth 4→2, dropout 0.1→0.4 (small-dataset regularisation)
    def __init__(self,clinical_dim,img_embed_dim=512,clin_embed_dim=512,
                 tab_depth=2,tab_heads=4,n_classes=NUM_CLASSES,dropout=0.4,
                 n_cross_heads=8,vit_pretrained=True):
        super().__init__()
        self.img_encoder =PretrainedViTEncoder(embed_dim=img_embed_dim,pretrained=vit_pretrained)
        self.clin_encoder=TabTransformerEncoder(input_dim=clinical_dim,embed_dim=clin_embed_dim,
                                                depth=tab_depth,n_heads=tab_heads,dropout=dropout)
        self.clin_proj=(nn.Linear(clin_embed_dim,img_embed_dim)
                        if clin_embed_dim!=img_embed_dim else nn.Identity())
        self.cross_attn =CrossAttentionBlock(img_embed_dim,n_cross_heads,dropout)
        # 3: additional Dropout before final linear (two-stage regularisation)
        self.classifier = nn.Sequential(
            nn.Linear(img_embed_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(p=dropout),       # dropout=0.4 from __init__
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(p=dropout),       # second stage
            nn.Linear(128, n_classes)
        )
    def forward(self,img,clin):
        ie=self.img_encoder(img); ce=self.clin_proj(self.clin_encoder(clin))
        return self.classifier(self.cross_attn(ie,ce))

print("Section 5: AttentionIntermediateFusionNet (Apex) defined.")


In [ ]:

# SECTION 6: BASELINES & ABLATION MODELS


class BaselineCNNModel(nn.Module):
    """Image-only baselines. DenseNet-121 primary (plan-aligned)."""
    def __init__(self,backbone='resnet50',in_channels=NUM_CHANNELS,n_classes=NUM_CLASSES,pretrained=True,dropout=0.3):
        super().__init__(); self.backbone_name=backbone
        if backbone=='resnet50':
            b=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            b.conv1=CNNEncoder._adapt_conv(b.conv1,in_channels,pretrained); fd=b.fc.in_features; b.fc=nn.Identity(); self.enc=b
        elif backbone=='densenet121':
            b=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
            b.features.conv0=CNNEncoder._adapt_conv(b.features.conv0,in_channels,pretrained); b.classifier=nn.Identity(); fd=1024; self.enc=b
        elif backbone=='densenet201':
            b=models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None)
            b.features.conv0=CNNEncoder._adapt_conv(b.features.conv0,in_channels,pretrained); b.classifier=nn.Identity(); fd=1920; self.enc=b
        elif backbone in ('efficientnet_b0','efficientnet_b3'):
            fd=1280 if backbone=='efficientnet_b0' else 1536
            b=(models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
               if backbone=='efficientnet_b0' else
               models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None))
            b.features[0][0]=CNNEncoder._adapt_conv(b.features[0][0],in_channels,pretrained); b.classifier=nn.Identity(); self.enc=b
        else: raise ValueError(backbone)
        self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(fd,256),nn.GELU(),nn.Dropout(dropout),nn.Linear(256,n_classes))
    def forward(self,img,clin=None):
        if 'densenet' in self.backbone_name: f=F.adaptive_avg_pool2d(self.enc.features(img),(1,1)).flatten(1)
        elif 'efficientnet' in self.backbone_name: f=self.enc.avgpool(self.enc.features(img)).flatten(1)
        else: f=self.enc(img)
        return self.head(f)


class ClinicalOnlyMLP(nn.Module):
    def __init__(self,clinical_dim,n_classes=NUM_CLASSES,dropout=0.3):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(clinical_dim,256),nn.BatchNorm1d(256),nn.GELU(),nn.Dropout(dropout),
                               nn.Linear(256,128),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(dropout),
                               nn.Linear(128,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,img,clin): return self.net(clin)

class ClinicalOnlyTabTransformer(nn.Module):
    # 3: dropout raised 0.1→0.4
    def __init__(self,clinical_dim,n_classes=NUM_CLASSES,dropout=0.4):
        super().__init__()
        self.enc=TabTransformerEncoder(input_dim=clinical_dim,embed_dim=64,dropout=dropout)
        self.head=nn.Linear(64,n_classes)
    def forward(self,img,clin): return self.head(self.enc(clin))

class CNNMLPConcatModel(nn.Module):
    """Intermediate-fusion baseline: DenseNet-121 + MLP concat (plan Section 6.1)."""
    def __init__(self,clinical_dim,n_classes=NUM_CLASSES,dropout=0.3):
        super().__init__()
        self.cnn=CNNEncoder('densenet121',embed_dim=512,dropout=dropout)
        self.mlp=MLPEncoder(clinical_dim,embed_dim=64,dropout=dropout)
        self.head=nn.Sequential(nn.Linear(576,256),nn.LayerNorm(256),nn.GELU(),nn.Dropout(dropout),nn.Linear(256,n_classes))
    def forward(self,img,clin): return self.head(torch.cat([self.cnn(img),self.mlp(clin)],-1))

#  Ablation Models ─
class AblationNoCrossAttn(nn.Module):
    # 3: dropout raised 0.1→0.4
    def __init__(self,clinical_dim,n_classes=NUM_CLASSES,dropout=0.4):
        super().__init__()
        self.ie=PretrainedViTEncoder(embed_dim=512); self.ce=TabTransformerEncoder(input_dim=clinical_dim,embed_dim=512)
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(dropout),nn.Linear(256,n_classes))
    def forward(self,img,clin): return self.head(torch.cat([self.ie(img),self.ce(clin)],-1))

class AblationNoTabTransformer(nn.Module):
    # 3: dropout raised 0.1→0.4
    def __init__(self,clinical_dim,n_classes=NUM_CLASSES,dropout=0.4):
        super().__init__()
        self.ie=PretrainedViTEncoder(embed_dim=512); self.ce=MLPEncoder(clinical_dim,embed_dim=512,dropout=dropout)
        self.ca=CrossAttentionBlock(512,8,dropout)
        self.head=nn.Sequential(nn.Linear(512,256),nn.GELU(),nn.Dropout(dropout),nn.Linear(256,n_classes))
    def forward(self,img,clin): return self.head(self.ca(self.ie(img),self.ce(clin)))

class AblationImageOnly(nn.Module):
    def __init__(self,clinical_dim,**kw):
        super().__init__(); self.m=AttentionIntermediateFusionNet(clinical_dim=clinical_dim,**kw)
    def forward(self,img,clin): return self.m(img,torch.zeros_like(clin))

class AblationClinicalOnly(nn.Module):
    def __init__(self,clinical_dim,**kw):
        super().__init__(); self.m=AttentionIntermediateFusionNet(clinical_dim=clinical_dim,**kw)
    def forward(self,img,clin): return self.m(torch.zeros_like(img),clin)

print("Section 6: Baselines and ablation models defined.")



In [ ]:

# SECTION 7: GRAD-CAM INTERPRETABILITY


class GradCAMViT:
    def __init__(self,model,device=DEVICE): self.model=model.to(device); self.device=device

    def _get_cam(self,img,clin,target_class=1):
        self.model.eval(); fs=[]; gs=[]
        # Hook the last attention block of the ViT
        enc=self.model.img_encoder
        if _TIMM_OK and hasattr(enc,'vit'): lb=enc.vit.blocks[-1]
        else: lb=enc._fb.tr.layers[-1]
        hf=lb.register_forward_hook(lambda m,i,o: fs.append(o.detach()))
        hb=lb.register_full_backward_hook(lambda m,gi,go: gs.append(go[0].detach()))
        with torch.enable_grad():
            it=img.to(self.device).requires_grad_(True); ct=clin.to(self.device)
            logits=self.model(it,ct); self.model.zero_grad(); logits[0,target_class].backward()
        hf.remove(); hb.remove()
        if not fs or not gs: return np.zeros((IMG_SIZE,IMG_SIZE))
        cam=F.relu((gs[0].mean(dim=-1,keepdim=True)*fs[0]).sum(dim=-1))[0,1:].cpu().numpy()
        n=int(math.sqrt(cam.shape[0])); cam=cam[:n*n].reshape(n,n)
        if cam.max()>0: cam/=cam.max()
        return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))/255.0

    def visualize_batch(self,img_b,clin_b,n_samples=5,save_path=None):
        N=min(n_samples,img_b.size(0)); fig,axes=plt.subplots(N,NUM_CHANNELS+1,figsize=(5*(NUM_CHANNELS+1),4*N))
        if N==1: axes=axes[np.newaxis,:]
        for i in range(N):
            hm=self._get_cam(img_b[i:i+1],clin_b[i:i+1]); inp=img_b[i].cpu().numpy()
            axes[i,0].imshow(hm,cmap='jet'); axes[i,0].set_title(f'CAM {i+1}'); axes[i,0].axis('off')
            for j,mt in enumerate(MAP_TYPES):
                axes[i,j+1].imshow(inp[j],cmap='gray'); axes[i,j+1].imshow(hm,cmap='jet',alpha=0.4)
                axes[i,j+1].set_title(mt); axes[i,j+1].axis('off')
        plt.suptitle('Grad-CAM: ViT Heatmaps on Orbscan',fontsize=14,fontweight='bold')
        plt.tight_layout()
        if save_path: plt.savefig(save_path,dpi=150,bbox_inches='tight')
        plt.close(); print(f"  [GradCAM] {save_path}")


class TabularGradCAM:
    def __init__(self,model,device=DEVICE): self.model=model.to(device); self.device=device

    def compute_importance(self,img,clin,feature_names,target_class=1):
        self.model.eval()
        with torch.enable_grad():
            cg=clin.to(self.device).requires_grad_(True); ig=img.to(self.device)
            logits=self.model(ig,cg); self.model.zero_grad(); logits[0,target_class].backward()
        imp=cg.grad.abs().mean(dim=0).cpu().numpy(); imp/=(imp.sum()+1e-8)
        n=min(len(feature_names),len(imp))
        return pd.DataFrame({'Feature':feature_names[:n],'Importance':imp[:n]}).sort_values('Importance',ascending=False).reset_index(drop=True)

    def plot_importance(self,df,save_path=None):
        fig,ax=plt.subplots(figsize=(10,max(5,len(df)*0.5)))
        ax.barh(df['Feature'],df['Importance'],color=plt.cm.viridis(np.linspace(0.2,0.9,len(df))))
        ax.set_xlabel('Normalized Importance'); ax.set_title('Clinical Feature Importance (TabularGradCAM)')
        ax.invert_yaxis(); plt.tight_layout()
        if save_path: plt.savefig(save_path,dpi=150,bbox_inches='tight')
        plt.close(); print(f"  [TabCAM] {save_path}")

print("Section 7: GradCAM classes defined.")


In [ ]:

# SECTION 8: PERFORMANCE EVALUATION METRICS

def compute_all_metrics(y_true,y_pred,y_prob):
    cv=confusion_matrix(y_true,y_pred).ravel()
    tn,fp,fn,tp=cv if len(cv)==4 else (0,0,0,0)
    return {
        'accuracy':          float(accuracy_score(y_true,y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true,y_pred)),
        'auc_roc':           float(roc_auc_score(y_true,y_prob)) if len(np.unique(y_true))>1 else 0.0,
        'auc_pr':            float(average_precision_score(y_true,y_prob)) if len(np.unique(y_true))>1 else 0.0,
        'sensitivity':       float(tp/(tp+fn+1e-8)),
        'specificity':       float(tn/(tn+fp+1e-8)),
        'precision':         float(precision_score(y_true,y_pred,zero_division=0)),
        'f1_score':          float(f1_score(y_true,y_pred,zero_division=0)),
        'mcc':               float(matthews_corrcoef(y_true,y_pred)),
    }


def bootstrap_metrics(y_true,y_pred,y_prob,n_iters=N_BOOTSTRAP,ci=0.95,seed=SEED):
    """
    Bootstrap 95% CI for all primary metrics.
    """
    rng=np.random.default_rng(seed); n=len(y_true); store=defaultdict(list)
    for _ in tqdm(range(n_iters),desc=f'  Bootstrap ({n_iters} iters)',leave=False):
        idx=rng.integers(0,n,size=n); yt,yp,ypr=y_true[idx],y_pred[idx],y_prob[idx]
        if len(np.unique(yt))<2: continue
        for k,v in compute_all_metrics(yt,yp,ypr).items(): store[k].append(v)
    a=1-ci; rows=[]
    for k,vals in store.items():
        arr=np.array(vals)
        rows.append({'metric':k,'mean':arr.mean(),'ci_lower':np.percentile(arr,100*a/2),'ci_upper':np.percentile(arr,100*(1-a/2))})
    return pd.DataFrame(rows).set_index('metric')


def format_metrics_table(bs_df):
    return pd.DataFrame.from_dict(
        {m:f"{r['mean']:.4f} [{r['ci_lower']:.4f} - {r['ci_upper']:.4f}]"
         for m,r in bs_df.iterrows()},
        orient='index', columns=['Mean [95% CI]'])


def net_reclassification_improvement(y_true,y_prob_new,y_prob_old,threshold=0.5):
    ev=y_true==1; nev=y_true==0
    up=(y_prob_new>threshold)&(y_prob_old<=threshold)
    dn=(y_prob_new<=threshold)&(y_prob_old>threshold)
    ne=float(up[ev].mean()-dn[ev].mean()); nn=float(dn[nev].mean()-up[nev].mean())
    return {'nri_events':ne,'nri_nonevents':nn,'nri_overall':ne+nn}


def generate_nri_table(apex_result:dict, all_results:dict, save_path=None) -> pd.DataFrame:
    """
    Plan Section 8 requires Clinical Utility: NRI for each comparator.
    Was previously only computed for ResNet-50.
    """
    rows=[]
    ap=apex_result['y_prob']; at=apex_result['y_true']
    for name,res in all_results.items():
        if name=='Apex_FusionTransformer': continue
        nri=net_reclassification_improvement(at,ap,res['y_prob'])
        rows.append({'Comparator':name,**nri})
    df=pd.DataFrame(rows)
    print("\n  NRI Table -- Apex_FusionTransformer vs All Baselines")
    print("  " + "-"*70); print(df.to_string(index=False))
    if save_path: df.to_csv(save_path,index=False); print(f"  [NRI] Saved: {save_path}")
    return df


def decision_curve_analysis(y_true,y_prob,thresholds=np.linspace(0.01,0.99,99),save_path=None):
    n,prev=len(y_true),y_true.mean(); rows=[]
    for t in thresholds:
        yp=(y_prob>=t).astype(int); tp=((yp==1)&(y_true==1)).sum(); fp=((yp==1)&(y_true==0)).sum()
        rows.append({'threshold':t,'net_benefit':tp/n-fp/n*(t/(1-t+1e-8))})
    dca=pd.DataFrame(rows)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.plot(dca['threshold'],dca['net_benefit'],'b-',lw=2,label='Model')
    ax.axhline(0,color='gray',ls='--',label='Treat None')
    ax.plot(thresholds,[prev-(1-prev)*t/(1-t+1e-8) for t in thresholds],'r--',label='Treat All')
    ax.set_xlabel('Threshold Probability'); ax.set_ylabel('Net Benefit')
    ax.set_title('Decision Curve Analysis'); ax.legend(); plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); return dca


def plot_roc_pr_curves(results_dict,save_path=None):
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,6))
    colors=plt.cm.tab10(np.linspace(0,0.9,len(results_dict)))
    for (nm,(yt,yp)),c in zip(results_dict.items(),colors):
        fpr,tpr,_=roc_curve(yt,yp); a1.plot(fpr,tpr,lw=2,color=c,label=f"{nm}({roc_auc_score(yt,yp):.3f})")
        pr,rc,_=precision_recall_curve(yt,yp); a2.plot(rc,pr,lw=2,color=c,label=f"{nm}({average_precision_score(yt,yp):.3f})")
    a1.plot([0,1],[0,1],'k--',lw=1); a1.set(xlabel='FPR',ylabel='TPR',title='ROC'); a1.legend(fontsize=7)
    a2.set(xlabel='Recall',ylabel='Precision',title='PR Curves'); a2.legend(fontsize=7)
    plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close()


def print_confusion_matrix(y_true,y_pred,model_name):
    cm=confusion_matrix(y_true,y_pred); fig,ax=plt.subplots(figsize=(5,4))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax)
    ax.set_title(f'Confusion Matrix -- {model_name}'); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,f'cm_{model_name.replace(" ","_")}.png'),dpi=120); plt.close()

print(f"Section 8: Metrics engine defined. N_BOOTSTRAP={N_BOOTSTRAP}")
print("  generate_nri_table(): Apex vs ALL baselines")


In [ ]:

# TRAINING ENGINE


class EarlyStopping:
    def __init__(self,patience=PATIENCE,delta=1e-4):
        self.patience=patience; self.delta=delta; self.counter=0; self.best=np.inf; self.triggered=False
    def step(self,vl):
        if vl<self.best-self.delta: self.best=vl; self.counter=0
        else:
            self.counter+=1
            if self.counter>=self.patience: self.triggered=True
        return self.triggered

class FocalLoss(nn.Module):
    # 3: label_smoothing=0.1 added to reduce overconfidence on small dataset
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.a  = alpha
        self.g  = gamma
        self.ls = label_smoothing   # 0.1 per audit recommendation

    def forward(self, logits, targets):
        n_cls = logits.size(1)
        # Build soft targets: (1 - ls) at true class, ls/(n_cls-1) elsewhere
        with torch.no_grad():
            soft = torch.full_like(logits, self.ls / max(n_cls - 1, 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_prob = F.log_softmax(logits, dim=1)
        ce = -(soft * log_prob).sum(dim=1)          # smoothed cross-entropy per sample
        pt = torch.exp(-ce)
        focal_loss = self.a * (1 - pt) ** self.g * ce
        return focal_loss.mean()

def get_class_weights(labels):
    c=np.bincount(labels); w=1/(c+1e-8); return torch.tensor(w/w.sum()*len(c),dtype=torch.float32,device=DEVICE)

_AMP=("cuda" if torch.cuda.is_available() else "cpu")

def train_one_epoch(model,loader,optimizer,criterion,scaler,device):
    model.train(); tl=cr=tot=0
    for batch in loader:
        img=batch[0].to(device); clin=batch[1].to(device); labels=batch[2].to(device)
        optimizer.zero_grad()
        with autocast(device_type=_AMP): logits=model(img,clin); loss=criterion(logits,labels)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        with torch.no_grad(): cr+=(logits.detach().argmax(-1)==labels).sum().item()
        tl+=loss.item()*labels.size(0); tot+=labels.size(0)
    return tl/tot, cr/tot

@torch.no_grad()
def evaluate(model,loader,criterion,device):
    model.eval(); tl=cr=tot=0; yt,yp,ypr=[],[],[]
    for batch in loader:
        img=batch[0].to(device); clin=batch[1].to(device); labels=batch[2].to(device)
        with autocast(device_type=_AMP): logits=model(img,clin); loss=criterion(logits,labels)
        probs=F.softmax(logits,-1)[:,1]
        cr+=(logits.argmax(-1)==labels).sum().item(); tl+=loss.item()*labels.size(0); tot+=labels.size(0)
        yt.extend(labels.cpu().numpy()); yp.extend(logits.argmax(-1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
    return tl/tot, cr/tot, np.array(yt), np.array(yp), np.array(ypr)

def train_model(model,train_loader,val_loader,model_name,n_epochs=NUM_EPOCHS,
                lr=LR,weight_decay=WEIGHT_DECAY,use_focal=True,patience=PATIENCE,device=DEVICE):
    model=model.to(device); opt=optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=1e-6)
    scaler=GradScaler(_AMP); es=EarlyStopping(patience=patience)
    crit=FocalLoss() if use_focal else nn.CrossEntropyLoss()
    history=dict(train_loss=[],val_loss=[],train_acc=[],val_acc=[],val_auc=[])
    bvl=np.inf; bst=None
    np_=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  {'-'*56}\n  Training: {model_name}  ({np_:,} params)\n  {'-'*56}")
    for ep in range(1,n_epochs+1):
        t0=time.time(); trl,tra=train_one_epoch(model,train_loader,opt,crit,scaler,device)
        vll,vla,yt,yp,ypr=evaluate(model,val_loader,crit,device)
        vauc=roc_auc_score(yt,ypr) if len(np.unique(yt))>1 else 0.0; sched.step()
        for k,v in zip(["train_loss","val_loss","train_acc","val_acc","val_auc"],[trl,vll,tra,vla,vauc]):
            history[k].append(v)
        print(f"  Ep {ep:03d}/{n_epochs} | TrL={trl:.4f} TrA={tra:.4f} | VlL={vll:.4f} VlA={vla:.4f} AUC={vauc:.4f} | {time.time()-t0:.1f}s")
        if vll<bvl:
            bvl=vll; bst=copy.deepcopy(model.state_dict())
            torch.save({"state_dict":bst,"epoch":ep},os.path.join(OUTPUT_DIR,f"{model_name}_best.pth"))
        if es.step(vll): print(f"  Early stop @ ep {ep}"); break
    if bst: model.load_state_dict(bst)
    _plot_curves(history,model_name); return model,history

def _plot_curves(h,name):
    ep=range(1,len(h["train_loss"])+1); fig,axes=plt.subplots(1,3,figsize=(18,5))
    axes[0].plot(ep,h["train_loss"],label="Train"); axes[0].plot(ep,h["val_loss"],label="Val"); axes[0].set_title(f"{name}-Loss"); axes[0].legend()
    axes[1].plot(ep,h["train_acc"],label="Train"); axes[1].plot(ep,h["val_acc"],label="Val"); axes[1].set_title(f"{name}-Acc"); axes[1].legend()
    axes[2].plot(ep,h["val_auc"],color="green"); axes[2].set_title(f"{name}-ValAUC")
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,f"{name}_curves.png"),dpi=100); plt.close()

print("Training engine: mixed-precision + CosineAnnealingLR + FocalLoss.")


In [ ]:

# SECTION 6.4: MISSING MODALITY ROBUSTNESS + DATA EFFICIENCY


def modality_dropout_train(model,train_loader,val_loader,model_name,p_drop=0.3,**kw):

    class ModalityDropoutWrapper(nn.Module):
        def __init__(self,base,p): super().__init__(); self.base=base; self.p=p
        def forward(self,img,clin):
            if self.training:
                r=random.random()
                if   r < self.p:         img =torch.zeros_like(img)    # image-only dropped
                elif r < 2*self.p:       clin=torch.zeros_like(clin)   # clinical-only dropped
                # else r >= 2p: both kept -- never zero both simultaneously
            return self.base(img,clin)

    wrapped=ModalityDropoutWrapper(model,p_drop)
    trained,hist=train_model(wrapped,train_loader,val_loader,model_name=model_name,**kw)
    return trained.base,hist


@torch.no_grad()
def evaluate_missing_modality(model,loader,missing,device=DEVICE):
    model.eval(); yt,yp,ypr=[],[],[]
    for batch in loader:
        img,clin,labels=batch[0].to(device),batch[1].to(device),batch[2].to(device)
        if missing=='image':    img =torch.zeros_like(img)
        elif missing=='clinical': clin=torch.zeros_like(clin)
        logits=model(img,clin); probs=F.softmax(logits,-1)[:,1]
        yt.extend(labels.cpu().numpy()); yp.extend(logits.argmax(-1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
    m=compute_all_metrics(np.array(yt),np.array(yp),np.array(ypr))
    print(f"  [Missing={missing}] AUC={m['auc_roc']:.4f} F1={m['f1_score']:.4f} Sens={m['sensitivity']:.4f} Spec={m['specificity']:.4f}")
    return m


def data_efficiency_test(model_class,model_kwargs,full_train_dataset,val_loader,test_loader,
                          fractions=[0.1,0.25,0.5,1.0],model_name='model'):
    """
    StratifiedShuffleSplit ensures KC:Normal ratio is preserved
    at every fraction. Old np.random.choice could yield 90% KC at 10% data.
    """
    results=[]; n=len(full_train_dataset)
    print(f"  [DataEff] Extracting labels for stratification ({n} samples)...")
    labels=np.array([full_train_dataset[i][2].item() for i in range(n)])
    for frac in fractions:
        set_seed(SEED)
        if frac>=1.0:
            subset=full_train_dataset
        else:
            sss=StratifiedShuffleSplit(n_splits=1,test_size=1-frac,random_state=SEED)
            idx,_=next(sss.split(np.zeros(n),labels))
            print(f"  [DataEff] frac={frac:.2f} -> {len(idx)} samples | KC={labels[idx].sum()} N={len(idx)-labels[idx].sum()}")
            subset=Subset(full_train_dataset,idx)
        loader=DataLoader(subset,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,drop_last=True)
        m_=model_class(**model_kwargs).to(DEVICE)
        m_,_=train_model(m_,loader,val_loader,model_name=f'{model_name}_frac{frac}',n_epochs=30,patience=5)
        _,_,yt,yp,ypr=evaluate(m_,test_loader,FocalLoss(),DEVICE)
        met=compute_all_metrics(yt,yp,ypr); met.update({'fraction':frac,'n_samples':len(subset)}); results.append(met)
        print(f"  [DataEff] frac={frac:.2f} n={len(subset):4d} AUC={met['auc_roc']:.4f} F1={met['f1_score']:.4f}")
    df=pd.DataFrame(results)
    fig,ax=plt.subplots(figsize=(7,4))
    ax.plot(df['fraction'],df['auc_roc'],'b-o',label='AUC-ROC'); ax.plot(df['fraction'],df['f1_score'],'r-s',label='F1')
    ax.set_xlabel('Training Fraction (Stratified)'); ax.set_ylabel('Score')
    ax.set_title('Data Efficiency -- Stratified Subsets'); ax.legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,f'{model_name}_data_efficiency.png'),dpi=120); plt.close()
    return df


#  Preprocessing ablation builders 
def build_no_norm_dataset(df,clin_arr,augment=False):
    """Ablation: skip Z-score normalisation."""
    return OrbscanDataset(df,clin_arr,transform=get_train_transforms() if augment else get_val_transforms(),augment=augment)

def build_no_reg_dataset(df,clin_arr,norm_means,norm_stds,augment=False):
    """Ablation: skip affine registration."""
    return OrbscanDataset(df,clin_arr,transform=get_train_transforms() if augment else get_val_transforms(),
                          augment=augment,registrar=None,normalizer_means=norm_means,normalizer_stds=norm_stds)

def build_no_aug_dataset(df,clin_arr,registrar,norm_means,norm_stds):
    """
    Augmentation ablation.
    Plan Section 6.3 requires: 'Omit critical preprocessing steps -- augmentation'.
    Returns dataset with augment=False and val_transforms only
    (no flips/affine/elastic/noise).
    """
    return OrbscanDataset(df,clin_arr,
        transform=get_val_transforms(),  # deterministic resize only
        augment=False,                   # disables elastic + Gaussian noise
        registrar=registrar,
        normalizer_means=norm_means,normalizer_stds=norm_stds)




In [ ]:

# SECTION 9: PIPELINE HELPER FUNCTIONS


def fit_per_channel_registrars(train_df, n_samples=200):
    """
    One AffineRegistrar per MAP_TYPE.
    Previously one Anterior-only registrar was applied to all 4 channels.
    Pachymetry (um), Axial and Posterior have different intensity profiles
    -- cross-channel phase-shift templates are geometrically incorrect.
    """
    regs = {}
    print(f"  [Registration] Fitting per-channel registrars (n_samples={n_samples})...")
    for mt in MAP_TYPES:
        maps = []
        for _, row in train_df.head(n_samples).iterrows():
            p = row.get(mt, None)
            if p and isinstance(p, str) and os.path.exists(p):
                img = Image.open(p).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                maps.append(np.array(img, dtype=np.float32) / 255.0)
        reg = AffineRegistrar()
        if maps:
            reg.fit(np.stack(maps, 0))
            print(f"    [{mt:11s}] template fitted on {len(maps)} images.")
        else:
            print(f"    [{mt:11s}] no images found -- skipping registration for this channel.")
        regs[mt] = reg
    return regs


def build_normalizer(stats_ds, n_samples=500):
    """
    Stats_ds MUST be non-augmented (augment=False, val_transforms).
    Random flips alter spatial means; affine rescaling biases intensity stds.
    """
    print("  [Normalizer] Computing stats on NON-AUGMENTED training data...")
    loader = DataLoader(stats_ds, batch_size=32, shuffle=False, num_workers=0)
    sums = torch.zeros(NUM_CHANNELS)
    sq   = torch.zeros(NUM_CHANNELS)
    cnt  = 0
    for batch in loader:
        imgs = batch[0]
        sums += imgs.mean(dim=(0,2,3)) * imgs.size(0)
        sq   += (imgs**2).mean(dim=(0,2,3)) * imgs.size(0)
        cnt  += imgs.size(0)
        if cnt >= n_samples:
            break
    means = (sums / cnt).numpy()
    var   = torch.clamp(sq / cnt - torch.tensor(means, dtype=torch.float32)**2, min=1e-8)
    stds  = torch.sqrt(var).numpy() + 1e-8
    print(f"  Channel means : {np.round(means, 4)}")
    print(f"  Channel stds  : {np.round(stds,  4)}")
    return means, stds


def build_dataloaders(tdf, vdf, tedf, ct, cv, cte,
                      norm_means=None, norm_stds=None, registrar=None,
                      missing_strategy='exclude', batch_size=BATCH_SIZE,
                      num_workers=2 if torch.cuda.is_available() else 0):
    tds  = OrbscanDataset(tdf,  ct,  transform=get_train_transforms(), augment=True,
                          registrar=registrar, normalizer_means=norm_means, normalizer_stds=norm_stds,
                          missing_strategy=missing_strategy)
    vds  = OrbscanDataset(vdf,  cv,  transform=get_val_transforms(),   augment=False,
                          registrar=registrar, normalizer_means=norm_means, normalizer_stds=norm_stds,
                          missing_strategy=missing_strategy)
    teds = OrbscanDataset(tedf, cte, transform=get_val_transforms(),   augment=False,
                          registrar=registrar, normalizer_means=norm_means, normalizer_stds=norm_stds,
                          missing_strategy=missing_strategy)
    labels  = tdf['label'].values
    sw      = get_class_weights(labels)[labels]
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
    nw      = num_workers
    TL  = DataLoader(tds,  batch_size=batch_size, sampler=sampler,    num_workers=nw, pin_memory=True, drop_last=True)
    VL  = DataLoader(vds,  batch_size=batch_size, shuffle=False,      num_workers=nw, pin_memory=True)
    TEL = DataLoader(teds, batch_size=batch_size, shuffle=False,      num_workers=nw, pin_memory=True)
    return TL, VL, TEL, tds, vds, teds


def _store(results, name, loader, model, device=DEVICE):
    _, _, yt, yp, ypr = evaluate(model, loader, FocalLoss(), device)
    results[name] = {'y_true': yt, 'y_pred': yp, 'y_prob': ypr}
    m = compute_all_metrics(yt, yp, ypr)
    print(f"  {name:45s} | AUC={m['auc_roc']:.4f}  F1={m['f1_score']:.4f}  "
          f"Sens={m['sensitivity']:.4f}  Spec={m['specificity']:.4f}")
    return m

print("Pipeline helper functions defined.")


In [ ]:

# DATA LOADING + CLINICAL PREPROCESSING



print("  KERATOCONUS MULTIMODAL FUSION ")
print("  Scholar: Sreedhar  |  Dataset: Zenodo 17127265")


#  Step 1: Scan images and split dataset ─
print("\n[STEP 1] Dataset analysis and stratified split...")
auditor = DataAuditor(missing_strategy='exclude')
full_df, train_df, val_df, test_df = auditor.run()

print(f"\n  Dataset summary (patient-level split — 1):")
print(f"    Total eye-rows           : {len(full_df)}")
print(f"    Unique patients (total)  : {full_df['patient_code'].nunique()}")
print(f"    Train (70%) eye-rows     : {len(train_df):4d} | "
      f"patients: {train_df['patient_code'].nunique()}")
print(f"    Val   (15%) eye-rows     : {len(val_df):4d} | "
      f"patients: {val_df['patient_code'].nunique()}")
print(f"    Test  (15%) eye-rows     : {len(test_df):4d} | "
      f"patients: {test_df['patient_code'].nunique()}")
# Verify zero patient overlap — hard check
_tr_pts = set(train_df['patient_code'])
_vl_pts = set(val_df['patient_code'])
_te_pts = set(test_df['patient_code'])
assert not (_tr_pts & _vl_pts), 'LEAK: train/val patient overlap!'
assert not (_tr_pts & _te_pts), 'LEAK: train/test patient overlap!'
assert not (_vl_pts & _te_pts), 'LEAK: val/test patient overlap!'
print("    Patient-level isolation verified ✓  (no cross-partition leakage)")

#  Step 2: Clinical preprocessing -- fit on TRAIN ONLY 
print("\n[STEP 2] Clinical preprocessing (fit on train only)...")
clin_pp    = ClinicalPreprocessor(imputer='knn', scaler='minmax',
                                   cat_encoding='onehot', dim_reduction='none')
clin_train = clin_pp.fit_transform(train_df)   # fit + transform on train
clin_val   = clin_pp.transform(val_df)          # transform only on val
clin_test  = clin_pp.transform(test_df)         # transform only on test

CDIM = clin_train.shape[1]
print(f"\n  Clinical feature dimension (CDIM) : {CDIM}")
print(f"  clin_train shape                  : {clin_train.shape}")
print(f"  clin_val   shape                  : {clin_val.shape}")
print(f"  clin_test  shape                  : {clin_test.shape}")


In [ ]:

# EXECUTE  STEP 2.1 + 3: PER-CHANNEL REGISTRATION + NORMALIZER + DATALOADERS


#  Step 2.1: Per-channel registrars 
print("[STEP 2.1] Fitting per-channel AffineRegistrars ...")
registrars = fit_per_channel_registrars(train_df, n_samples=200)

#  Step 3a: Compute normalizer stats on NON-AUGMENTED dataset 
print("\n[STEP 3a] Channel normalizer -- NON-AUGMENTED stats_ds ...")
stats_ds = OrbscanDataset(
    train_df, clin_train,
    transform=get_val_transforms(),  
    augment=False,
    registrar=registrars,
    normalizer_means=None,          
    normalizer_stds=None
)
norm_means, norm_stds = build_normalizer(stats_ds, n_samples=500)
del stats_ds  

#  Step 3b: Build main dataloaders 
print("\n[STEP 3b] Building dataloaders (augmented train, clean val/test)...")
TL, VL, TEL, train_ds_full, _, _ = build_dataloaders(
    train_df, val_df, test_df,
    clin_train, clin_val, clin_test,
    norm_means=norm_means,
    norm_stds=norm_stds,
    registrar=registrars
)

print(f"\n  Dataloader sizes:")
print(f"    Train batches : {len(TL)}  (batch={BATCH_SIZE}, WeightedSampler)")
print(f"    Val   batches : {len(VL)}")
print(f"    Test  batches : {len(TEL)}")

#  Global results dict (populated incrementally across cells) 
ALL_RESULTS  = {}   # Dict[model_name -> {y_true, y_pred, y_prob}]
ALL_HISTORY  = {}   # Dict[model_name -> training history]


In [ ]:

# EXECUTE  STEP 5: APEX MODEL TRAINING
# PretrainedViT-B/16 (timm, 4-ch) + TabTransformer-CLS + CrossAttention



print("  [STEP 5] APEX MODEL: AttentionIntermediateFusionNet")
print("  Encoders : PretrainedViTEncoder (timm ViT-B/16)  +  TabTransformerEncoder (CLS-token)")
print("  Fusion   : Bidirectional CrossAttentionBlock")
print("  Classifier: 2-layer MLP + Softmax")


apex_model = AttentionIntermediateFusionNet(clinical_dim=CDIM)

apex_model, apex_history = train_model(
    apex_model, TL, VL,
    model_name='Apex_FusionTransformer',
    n_epochs=NUM_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE
)

_store(ALL_RESULTS, 'Apex_FusionTransformer', TEL, apex_model)
ALL_HISTORY['Apex_FusionTransformer'] = apex_history

print("\n  Apex model training complete.")


In [ ]:

# EXECUTE  STEP 6.1: SOTA BASELINES + UNIMODAL BASELINES
# DenseNet-201, ResNet-50, EfficientNet = extended comparison



print("  [STEP 6.1] SOTA & UNIMODAL BASELINES")
print("  DenseNet-121: PRIMARY")


#  Image-only CNN baselines 
for backbone in ['densenet121', 'resnet50', 'densenet201', 'efficientnet_b0', 'efficientnet_b3']:
    nm = f'Baseline_{backbone}'
    print(f"\n  {nm}")
    m = BaselineCNNModel(backbone=backbone)
    m, h = train_model(m, TL, VL, model_name=nm,
                       n_epochs=NUM_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE)
    _store(ALL_RESULTS, nm, TEL, m)
    ALL_HISTORY[nm] = h
    del m  

#  Clinical-only baselines ─
for cls_, nm in [
    (ClinicalOnlyMLP,            'Baseline_ClinicalMLP'),
    (ClinicalOnlyTabTransformer, 'Baseline_ClinicalTabTransformer'),
]:
    print(f"\n  {nm}")
    m = cls_(CDIM)
    m, h = train_model(m, TL, VL, model_name=nm,
                       n_epochs=NUM_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE)
    _store(ALL_RESULTS, nm, TEL, m)
    ALL_HISTORY[nm] = h
    del m

#  Intermediate fusion baseline (CNN + MLP concat) ─
print("\n  Baseline_CNNMLPConcat")
m = CNNMLPConcatModel(CDIM)
m, h = train_model(m, TL, VL, model_name='Baseline_CNNMLPConcat',
                   n_epochs=NUM_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE)
_store(ALL_RESULTS, 'Baseline_CNNMLPConcat', TEL, m)
ALL_HISTORY['Baseline_CNNMLPConcat'] = h
del m

print("\n  Section 6.1 baselines complete.")


In [ ]:

# EXECUTE  STEP 6.2: FUSION STRATEGY COMPARISON
# Early | Late (avg/vote/meta) | Intermediate | Hybrid
# Same encoders (PretrainedViT + TabTransformer) across all variants



print("  [STEP 6.2] FUSION STRATEGY COMPARISON")


fusion_configs = [
    ('early',        'average'),
    ('late',         'average'),
    ('late',         'voting'),
    ('late',         'meta'),
    ('intermediate', 'average'),
    ('hybrid',       'average'),
]

for ft, lm in fusion_configs:
    nm = f'Fusion_{ft}' + (f'_{lm}' if ft == 'late' else '')
    print(f"\n  {nm}  (fusion={ft}, combine={lm})")

    ie = PretrainedViTEncoder(embed_dim=512)
    ce = TabTransformerEncoder(input_dim=CDIM, embed_dim=512)

    kw = dict(
        fusion_type=ft,
        img_encoder=ie,
        clin_encoder=ce,
        img_embed_dim=512,
        clin_embed_dim=512,
        late_combine=lm,
    )
    if ft == 'early':
        kw['img_raw_dim']  = NUM_CHANNELS * IMG_SIZE * IMG_SIZE
        kw['clin_raw_dim'] = CDIM

    fm = FusionModule(**kw)
    fm, h = train_model(fm, TL, VL, model_name=nm,
                        n_epochs=NUM_EPOCHS, lr=LR,
                        weight_decay=WEIGHT_DECAY, patience=PATIENCE)
    _store(ALL_RESULTS, nm, TEL, fm)
    ALL_HISTORY[nm] = h
    del fm, ie, ce




In [ ]:

# EXECUTE  STEP 6.3: ABLATION STUDIES
# Component ablation: no cross-attn, no TabTransformer, image-only, clinical-only
# Preprocessing ablation: no normalisation, no registration




print(" ABLATION STUDIES")



component_ablations = [
    (AblationImageOnly,         'Ablation_ImageOnly'),
    (AblationClinicalOnly,      'Ablation_ClinicalOnly'),
    (AblationNoCrossAttn,       'Ablation_NoCrossAttn'),
    (AblationNoTabTransformer,  'Ablation_NoTabTransformer'),
]

for cls_, nm in component_ablations:
    print(f"\n  {nm}")
    m = cls_(clinical_dim=CDIM)
    m, h = train_model(m, TL, VL, model_name=nm,
                       n_epochs=NUM_EPOCHS, lr=LR,
                       weight_decay=WEIGHT_DECAY, patience=PATIENCE)
    _store(ALL_RESULTS, nm, TEL, m)
    ALL_HISTORY[nm] = h
    del m

print("\n  Ablation_NoNorm  (skip intensity normalisation)")
tN, vN, teN, _, _, _ = build_dataloaders(
    train_df, val_df, test_df,
    clin_train, clin_val, clin_test,
    norm_means=None, norm_stds=None,  
    registrar=registrars
)
mN = AttentionIntermediateFusionNet(clinical_dim=CDIM)
mN, h = train_model(mN, tN, vN, model_name='Ablation_NoNorm',
                    n_epochs=NUM_EPOCHS, lr=LR,
                    weight_decay=WEIGHT_DECAY, patience=PATIENCE)
_store(ALL_RESULTS, 'Ablation_NoNorm', teN, mN)
ALL_HISTORY['Ablation_NoNorm'] = h
del mN, tN, vN, teN


print("\n  Ablation_NoRegistration  (skip affine registration)")
tR, vR, teR, _, _, _ = build_dataloaders(
    train_df, val_df, test_df,
    clin_train, clin_val, clin_test,
    norm_means=norm_means, norm_stds=norm_stds,
    registrar=None        
)
mR = AttentionIntermediateFusionNet(clinical_dim=CDIM)
mR, h = train_model(mR, tR, vR, model_name='Ablation_NoRegistration',
                    n_epochs=NUM_EPOCHS, lr=LR,
                    weight_decay=WEIGHT_DECAY, patience=PATIENCE)
_store(ALL_RESULTS, 'Ablation_NoRegistration', teR, mR)
ALL_HISTORY['Ablation_NoRegistration'] = h
del mR, tR, vR, teR


print("\n  Ablation_NoAugmentation")
print("  Training without any data augmentation (no flips/affine/elastic/noise)")
no_aug_ds = build_no_aug_dataset(
    train_df, clin_train, registrars, norm_means, norm_stds
)
no_aug_loader = DataLoader(
    no_aug_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2 if torch.cuda.is_available() else 0,
    drop_last=True
)
mA = AttentionIntermediateFusionNet(clinical_dim=CDIM)
mA, h = train_model(mA, no_aug_loader, VL, model_name='Ablation_NoAugmentation',
                    n_epochs=NUM_EPOCHS, lr=LR,
                    weight_decay=WEIGHT_DECAY, patience=PATIENCE)
_store(ALL_RESULTS, 'Ablation_NoAugmentation', TEL, mA)
ALL_HISTORY['Ablation_NoAugmentation'] = h
del mA, no_aug_ds, no_aug_loader




In [ ]:

# EXECUTE  STEP 6.4: MISSING-MODALITY ROBUSTNESS + DATA EFFICIENCY



print("MISSING MODALITY ROBUSTNESS + DATA EFFICIENCY")


print("\n  Training Apex with MUTUALLY-EXCLUSIVE modality dropout (p=0.3)...")

apex_db = AttentionIntermediateFusionNet(clinical_dim=CDIM)
apex_dropout, _ = modality_dropout_train(
    apex_db, TL, VL,
    model_name='Apex_ModalityDropout',
    p_drop=0.3,
    n_epochs=NUM_EPOCHS, lr=LR,
    weight_decay=WEIGHT_DECAY, patience=PATIENCE
)
_store(ALL_RESULTS, 'Apex_ModalityDropout', TEL, apex_dropout)

print("\n  Missing modality evaluation on TEST set:")
print(f"  {'Model':<30s} {'Missing':<12s} {'AUC':>7s} {'F1':>7s} {'Sens':>7s} {'Spec':>7s}")


missing_robustness = {}
for mdl, nm in [(apex_model, 'Apex_Standard'), (apex_dropout, 'Apex_Dropout')]:
    for miss in ['image', 'clinical']:
        key = f'{nm}_missing_{miss}'
        res = evaluate_missing_modality(mdl, TEL, miss)
        missing_robustness[key] = res

print("\n  Data efficiency test (StratifiedShuffleSplit)...")
print("  Fractions: 10%, 25%, 50%, 100%")

data_eff_df = data_efficiency_test(
    model_class=AttentionIntermediateFusionNet,
    model_kwargs={'clinical_dim': CDIM},
    full_train_dataset=train_ds_full,
    val_loader=VL,
    test_loader=TEL,
    fractions=[0.10, 0.25, 0.50, 1.0],
    model_name='Apex'
)
data_eff_df.to_csv(os.path.join(OUTPUT_DIR, 'data_efficiency.csv'), index=False)

print("\n  Data Efficiency Results:")
print(data_eff_df[['fraction', 'n_samples', 'auc_roc', 'f1_score',
                   'sensitivity', 'specificity']].to_string(index=False))




In [ ]:

# EXECUTE  STEP 7: GRAD-CAM INTERPRETABILITY
# Visual heatmaps on ViT attention maps + tabular feature importance bar chart



print(" GRAD-CAM INTERPRETABILITY")


apex_model.eval()
sample_imgs, sample_clins = [], []
for batch in TEL:
    sample_imgs.append(batch[0])
    sample_clins.append(batch[1])
    if sum(x.size(0) for x in sample_imgs) >= 5:
        break
sample_imgs  = torch.cat(sample_imgs)[:5]
sample_clins = torch.cat(sample_clins)[:5]

print(f"  Sample batch  -- images:{tuple(sample_imgs.shape)}  "
      f"clinical:{tuple(sample_clins.shape)}")

#  7a  ViT Grad-CAM heatmaps ─
print("\n  [7a] ViT Grad-CAM heatmaps on 5 test samples...")
gradcam_vit = GradCAMViT(apex_model, DEVICE)
gradcam_vit.visualize_batch(
    sample_imgs, sample_clins,
    n_samples=5,
    save_path=os.path.join(OUTPUT_DIR, 'gradcam_vit_5samples.png')
)

#  7b  Tabular Grad-CAM feature importance ─
print("\n  [7b] Tabular Grad-CAM clinical feature importance...")
n_num = len([c for c in NUMERICAL_FEATURES if c in train_df.columns])
n_cat = CDIM - n_num
feature_names = (
    [c for c in NUMERICAL_FEATURES if c in train_df.columns] +
    (['gender_female', 'gender_male'] if n_cat == 2 else
     ['gender'] if n_cat == 1 else
     [f'cat_{i}' for i in range(n_cat)])
)

tcam = TabularGradCAM(apex_model, DEVICE)
imp_df = tcam.compute_importance(
    sample_imgs[:1].to(DEVICE),
    sample_clins[:1].to(DEVICE),
    feature_names=feature_names
)
tcam.plot_importance(
    imp_df,
    save_path=os.path.join(OUTPUT_DIR, 'tabular_gradcam_importance.png')
)

imp_df.to_csv(os.path.join(OUTPUT_DIR, 'tabular_feature_importance.csv'), index=False)

print("\n  Top-10 Clinical Features by Grad-CAM Importance:")
print(imp_df.head(10).to_string(index=False))




In [ ]:

# EXECUTE  STEP 8: COMPREHENSIVE METRICS + BOOTSTRAP 95% CIs + NRI TABLE



print(f" METRICS + BOOTSTRAP 95% CIs (N={N_BOOTSTRAP}) + NRI TABLE")


summary_rows = []

for model_name, res in ALL_RESULTS.items():
    yt, yp, ypr = res['y_true'], res['y_pred'], res['y_prob']

    base_m = compute_all_metrics(yt, yp, ypr)
    base_m['model'] = model_name
    summary_rows.append(base_m)

    print(f"\n  {'='*62}")
    print(f"  Model : {model_name}")
    print(f"  {'='*62}")
    for k, v in base_m.items():
        if k != 'model':
            print(f"    {k:<28s}: {v:.4f}")


    print(f"\n  Bootstrap 95% CI  ({N_BOOTSTRAP} resamples):")
    bs_df  = bootstrap_metrics(yt, yp, ypr, n_iters=N_BOOTSTRAP)
    fmt_df = format_metrics_table(bs_df)
    print(fmt_df.to_string())
    fmt_df.to_csv(os.path.join(OUTPUT_DIR, f'ci_{model_name}.csv'))


    print_confusion_matrix(yt, yp, model_name)


print("  NRI TABLE -- Apex_FusionTransformer vs ALL Baselines")

if 'Apex_FusionTransformer' in ALL_RESULTS:
    nri_df = generate_nri_table(
        ALL_RESULTS['Apex_FusionTransformer'],
        ALL_RESULTS,
        save_path=os.path.join(OUTPUT_DIR, 'nri_apex_vs_all_baselines.csv')
    )

print("\n  Decision Curve Analysis (Apex)...")
decision_curve_analysis(
    ALL_RESULTS['Apex_FusionTransformer']['y_true'],
    ALL_RESULTS['Apex_FusionTransformer']['y_prob'],
    save_path=os.path.join(OUTPUT_DIR, 'decision_curve_analysis.png')
)

print("  ROC + PR curves (all models)...")
plot_roc_pr_curves(
    {nm: (r['y_true'], r['y_prob']) for nm, r in ALL_RESULTS.items()},
    save_path=os.path.join(OUTPUT_DIR, 'roc_pr_all_models.png')
)


summary_df = pd.DataFrame(summary_rows).set_index('model').round(4)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'summary_all_models.csv'))


print("  FINAL SUMMARY TABLE  (Test Set)")

cols = ['accuracy', 'balanced_accuracy', 'auc_roc', 'auc_pr',
        'sensitivity', 'specificity', 'f1_score', 'mcc']
print(summary_df[cols].sort_values('auc_roc', ascending=False).to_string())
print(f"\n  Saved to : {OUTPUT_DIR}/summary_all_models.csv")


In [ ]:

# EXECUTE  STEP 10: VISUALIZATION SUITE


from sklearn.calibration import calibration_curve

def _fig_path(fn):
    return os.path.join(OUTPUT_DIR, fn)


def plot_lr_schedule(n_epochs=NUM_EPOCHS, lr=LR, save_path=None):
    d = nn.Linear(1, 1)
    opt = optim.AdamW(d.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-6)
    lrs = []
    for _ in range(n_epochs):
        lrs.append(opt.param_groups[0]['lr']); sched.step()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, n_epochs+1), lrs, color='#4C72B0', lw=2)
    ax.fill_between(range(1, n_epochs+1), lrs, alpha=0.10, color='#4C72B0')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
    ax.set_title(f'CosineAnnealingLR  (start={lr}, T_max={n_epochs})',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [V17] {save_path}")
    plt.show(); plt.close()


def plot_radar_chart(results_dict, save_path=None):
    metrics_keys   = ['accuracy', 'auc_roc', 'sensitivity', 'specificity', 'f1_score', 'mcc']
    metrics_labels = ['Accuracy', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1', 'MCC']
    n    = len(metrics_keys)
    angs = [i / n * 2 * np.pi for i in range(n)] + [0]
    fig, ax = plt.subplots(figsize=(11, 11), subplot_kw=dict(polar=True))
    colors  = plt.cm.tab10(np.linspace(0, 0.9, max(len(results_dict), 1)))
    for (mn, res), c in zip(results_dict.items(), colors):
        m    = compute_all_metrics(res['y_true'], res['y_pred'], res['y_prob'])
        vals = [(m[k] + 1) / 2 if k == 'mcc' else m[k] for k in metrics_keys]
        vals = vals + [vals[0]]
        ax.plot(angs, vals, lw=2, color=c, label=mn.replace('_', ' '))
        ax.fill(angs, vals, alpha=0.05, color=c)
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angs[:-1]), metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title('Model Performance Radar', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.40, 1.18), fontsize=7.5)
    ax.grid(True, alpha=0.35); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [V10] {save_path}")
    plt.show(); plt.close()


def plot_ablation_study(results_dict, apex_name='Apex_FusionTransformer', save_path=None):
    akeys = [k for k in results_dict if k.startswith('Ablation')]
    if apex_name in results_dict:
        akeys = [apex_name] + akeys
    mps = ['auc_roc', 'f1_score', 'sensitivity', 'specificity', 'balanced_accuracy']
    data = {
        mn: {k: compute_all_metrics(results_dict[mn]['y_true'],
                                     results_dict[mn]['y_pred'],
                                     results_dict[mn]['y_prob'])[k]
             for k in mps}
        for mn in akeys
    }
    df = pd.DataFrame(data).T
    x  = np.arange(len(mps))
    w  = 0.80 / max(len(akeys), 1)
    fig, ax = plt.subplots(figsize=(15, 7))
    colors  = plt.cm.Set2(np.linspace(0, 1, len(akeys)))
    for i, (mn, row) in enumerate(df.iterrows()):
        bars = ax.bar(x + i * w, row[mps].values, w,
                      label=mn.replace('_', ' '),
                      color=colors[i], edgecolor='black', linewidth=0.4, alpha=0.88)
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.004,
                    f'{b.get_height():.3f}', ha='center', va='bottom',
                    fontsize=6.5, rotation=45)
    ax.set_xticks(x + w * len(akeys) / 2)
    ax.set_xticklabels([m.replace('_', '\n') for m in mps], fontsize=10)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.14)
    ax.set_title('Ablation Study -- Component Contribution\n'
                 '(incl. Augmentation Ablation )',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left', framealpha=0.9)
    ax.grid(axis='y', alpha=0.30); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [V6] {save_path}")
    plt.show(); plt.close()


def plot_bootstrap_ci(results_dict, metric='auc_roc', n_iters=N_BOOTSTRAP, save_path=None):
    rows = []
    for mn, res in results_dict.items():
        bs = bootstrap_metrics(res['y_true'], res['y_pred'], res['y_prob'],
                               n_iters=n_iters)
        if metric in bs.index:
            rows.append({'model': mn,
                         'mean': bs.loc[metric, 'mean'],
                         'lo':   bs.loc[metric, 'ci_lower'],
                         'hi':   bs.loc[metric, 'ci_upper']})
    df = pd.DataFrame(rows).sort_values('mean', ascending=True)
    fig, ax = plt.subplots(figsize=(11, max(5, len(df) * 0.50)))
    colors = ['crimson' if 'Apex_FusionTransformer' in r['model']
              else '#4C72B0' for _, r in df.iterrows()]
    y = np.arange(len(df))
    ax.barh(y, df['mean'].values,
            xerr=[df['mean'].values - df['lo'].values,
                  df['hi'].values - df['mean'].values],
            height=0.55, color=colors, alpha=0.82,
            error_kw={'elinewidth': 1.5, 'capsize': 4})
    ax.set_yticks(y)
    ax.set_yticklabels([m.replace('_', ' ') for m in df['model']], fontsize=8.5)
    ax.set_xlabel(f'{metric.upper()}  mean +/- 95% CI  '
                  f'({n_iters} bootstrap resamples)', fontsize=11)
    ax.set_title(f'Bootstrap 95% CI -- {metric.upper()}',
                 fontsize=12, fontweight='bold')
    ax.axvline(0.5, color='grey', ls='--', lw=1, label='Random classifier')
    ax.legend(); ax.grid(axis='x', alpha=0.30); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [V15] {save_path}")
    plt.show(); plt.close()


def plot_calibration(results_dict, save_path=None):
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(results_dict), 1)))
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
    for (mn, res), c in zip(results_dict.items(), colors):
        yt, ypr = res['y_true'], res['y_prob']
        if len(np.unique(yt)) < 2:
            continue
        fp, mp = calibration_curve(yt, ypr, n_bins=10, strategy='uniform')
        ax.plot(mp, fp, 's-', lw=1.8, color=c,
                label=mn.replace('_', ' '), alpha=0.85)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title('Calibration Diagram -- All Models',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.30); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [V11] {save_path}")
    plt.show(); plt.close()



print("  [STEP 10] VISUALIZATION SUITE")


def _safe(fn, *a, **kw):
    try:
        fn(*a, **kw)
    except Exception as e:
        print(f"  [VIZ WARN] {fn.__name__}: {e}")

print("\n  V17 -- Learning Rate Schedule")
_safe(plot_lr_schedule, NUM_EPOCHS, LR,
      save_path=_fig_path('V17_lr_schedule.png'))

print("\n  V10 -- Radar Chart (all models)")
_safe(plot_radar_chart, ALL_RESULTS,
      save_path=_fig_path('V10_radar_chart.png'))

print("\n  V6  -- Ablation Study Bar Chart")
_safe(plot_ablation_study, ALL_RESULTS,
      save_path=_fig_path('V6_ablation_study.png'))

print("\n  V15 -- Bootstrap CI (AUC-ROC)")
_safe(plot_bootstrap_ci, ALL_RESULTS, metric='auc_roc', n_iters=N_BOOTSTRAP,
      save_path=_fig_path('V15_bootstrap_ci_auc.png'))

print("\n  V15 -- Bootstrap CI (F1-Score)")
_safe(plot_bootstrap_ci, ALL_RESULTS, metric='f1_score', n_iters=N_BOOTSTRAP,
      save_path=_fig_path('V15_bootstrap_ci_f1.png'))

print("\n  V11 -- Calibration Diagram")
_safe(plot_calibration, ALL_RESULTS,
      save_path=_fig_path('V11_calibration.png'))

print("\n  Visualization suite complete.")
print(f"  All outputs saved to: {OUTPUT_DIR}")


In [ ]:
matplotlib.rcParams.update({
    'font.family'        : 'serif',
    'font.serif'         : ['DejaVu Serif', 'Times New Roman', 'Georgia', 'serif'],
    'font.size'          : 13,
    'axes.titlesize'     : 15,
    'axes.labelsize'     : 13,
    'xtick.labelsize'    : 11,
    'ytick.labelsize'    : 11,
    'legend.fontsize'    : 11,
    'figure.titlesize'   : 17,

    'figure.dpi'         : 120,
    'figure.constrained_layout.use' : True,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,

    'lines.linewidth'    : 2.2,
    'axes.grid'          : True,
    'grid.alpha'         : 0.30,
    'grid.linestyle'     : '--',
})
 
PALETTE = {
    'train'  : '#2563EB',   # blue
    'val'    : '#DC2626',   # red
    'apex'   : '#16A34A',   # green
    'normal' : '#6366F1',   # indigo
    'kc'     : '#F59E0B',   # amber
    'grad'   : 'inferno',   # Grad-CAM cmap
}
 
FUSION_LABEL = "Bidirectional Cross-Attention\nIntermediate Fusion (APEX)"
 

 

**CELL R-1  ·  Step 1 — Dataset Acquisition (Raw Sample Images)**

In [ ]:

def _show_raw_samples(df, n_patients=3, title="Step 1 · Dataset Acquisition — Raw Orbscan Maps",
                      save_path=None):
    """
    Display raw (un-preprocessed) 4-channel Orbscan images for n_patients.
    One row per patient-eye pair; columns = Anterior, Axial, Pachymetry, Posterior.
    """

    normals = df[df['label'] == 0].dropna(subset=MAP_TYPES).head(max(1, n_patients // 2))
    kcs     = df[df['label'] == 1].dropna(subset=MAP_TYPES).head(max(1, n_patients - len(normals)))
    samples = pd.concat([normals, kcs]).reset_index(drop=True).head(n_patients)
 
    n_rows = len(samples)
    fig, axes = plt.subplots(n_rows, len(MAP_TYPES),
                             figsize=(4.5 * len(MAP_TYPES), 3.5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
 
    for r, (_, row) in enumerate(samples.iterrows()):
        lbl = CLASS_NAMES[int(row['label'])]
        for c, mt in enumerate(MAP_TYPES):
            path = row.get(mt, None)
            ax   = axes[r, c]
            if path and os.path.exists(str(path)):
                img = np.array(Image.open(path).convert('L'))
                ax.imshow(img, cmap='gray', aspect='auto')
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center',
                        transform=ax.transAxes, color='red')
                ax.set_facecolor('#f0f0f0')
            ax.set_title(mt, fontweight='bold')
            ax.axis('off')
            if c == 0:
                ax.set_ylabel(f"P={row['patient_code']}\n({lbl})",
                              fontsize=10, rotation=0, labelpad=60, va='center')
 
    fig.suptitle(title, fontweight='bold', y=1.01)
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
 
_show_raw_samples(
    full_df,
    n_patients=4,
    title="Step 1 · Dataset Acquisition — Raw Orbscan Maps\n"
          f"(Zenodo ID 17127265 | Total pairs: {len(full_df)})",
    save_path=os.path.join(OUTPUT_DIR, 'R1_raw_samples.png')
)

**CELL R-2  ·  Preprocessing Visualisation + Class Distribution  **

In [ ]:
def _show_preprocessed_samples(df, n_patients=3,
                                title="Preprocessing — After Pipeline Transforms",
                                save_path=None):
    """
    Apply val_transforms and show the same patients as in R-1.
    """
    normals = df[df['label'] == 0].dropna(subset=MAP_TYPES).head(max(1, n_patients // 2))
    kcs     = df[df['label'] == 1].dropna(subset=MAP_TYPES).head(max(1, n_patients - len(normals)))
    samples = pd.concat([normals, kcs]).reset_index(drop=True).head(n_patients)
    tfm     = get_val_transforms()      
 
    n_rows = len(samples)
    fig, axes = plt.subplots(n_rows, len(MAP_TYPES),
                             figsize=(4.5 * len(MAP_TYPES), 3.5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
 
    for r, (_, row) in enumerate(samples.iterrows()):
        lbl = CLASS_NAMES[int(row['label'])]
        for c, mt in enumerate(MAP_TYPES):
            path = row.get(mt, None)
            ax   = axes[r, c]
            if path and os.path.exists(str(path)):
                raw = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
                # IsotropicResampler + val_transform
                resampled = IsotropicResampler(IMG_SIZE).resample(raw)
                pil_img   = Image.fromarray((resampled * 255).astype(np.uint8), mode='L')
                tensor    = tfm(pil_img)
                ax.imshow(tensor.squeeze().numpy(), cmap='gray', aspect='auto',
                          vmin=0, vmax=1)
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center',
                        transform=ax.transAxes, color='red')
                ax.set_facecolor('#f0f0f0')
            ax.set_title(mt, fontweight='bold')
            ax.axis('off')
            if c == 0:
                ax.set_ylabel(f"P={row['patient_code']}\n({lbl})",
                              fontsize=10, rotation=0, labelpad=60, va='center')
 
    fig.suptitle(title, fontweight='bold', y=1.01)
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
 
_show_preprocessed_samples(
    full_df, n_patients=4,
    title="Preprocessing — After IsotropicResampler + Val Transforms",
    save_path=os.path.join(OUTPUT_DIR, 'R2_preprocessed_samples.png')
)


**Class Distribution Before vs After Split**

In [ ]:
def _plot_class_distribution(save_path=None):
    splits = {
        'Full Dataset' : full_df,
        'Train (70%)'  : train_df,
        'Val   (15%)'  : val_df,
        'Test  (15%)'  : test_df,
    }
    normals = [s['label'].value_counts().get(0, 0) for s in splits.values()]
    kcs     = [s['label'].value_counts().get(1, 0) for s in splits.values()]
    x       = np.arange(len(splits))
    width   = 0.35
 
    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - width/2, normals, width, label='Normal',      color=PALETTE['normal'],  alpha=0.88)
    b2 = ax.bar(x + width/2, kcs,     width, label='Keratoconus', color=PALETTE['kc'],     alpha=0.88)
 
    for bar in b1 + b2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.5,
                f'{int(h)}', ha='center', va='bottom', fontsize=10)
 
    ax.set_xticks(x); ax.set_xticklabels(list(splits.keys()))
    ax.set_ylabel('Number of Samples')
    ax.set_title('Class Distribution — Before & After Stratified Split',
                 fontweight='bold')
    ax.legend()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
 
_plot_class_distribution(
    save_path=os.path.join(OUTPUT_DIR, 'R2_class_distribution.png')
)

**CELL R-3  ·  Smooth Training / Validation Curves (APEX)**

In [ ]:
def _smooth_curve(y, kind='spline', window=5):
    """
    Smooth a 1-D array.
      kind='spline'  → cubic B-spline interpolation (fewer control points)
      kind='ma'      → simple moving average (fallback for short sequences)
    Returns (x_new, y_new) ready for plotting.
    """
    x = np.arange(len(y))
    if len(y) < 6 or kind == 'ma':
        kernel = np.ones(window) / window
        y_s    = np.convolve(y, kernel, mode='same')
        return x, y_s

    k_pts  = min(len(y) - 1, max(4, len(y) // 3))
    xc     = np.linspace(0, len(y) - 1, k_pts)
    yc     = np.interp(xc, x, y)
    spline = make_interp_spline(xc, yc, k=3)
    x_new  = np.linspace(0, len(y) - 1, 300)
    return x_new, spline(x_new)
 
 
def plot_training_curves(history_dict, model_name='Apex_FusionTransformer',
                         save_path=None):
    """
    Plot Loss, Accuracy, and AUC curves for the given model's training history.
    Raw epoch values shown as faint markers; smooth spline drawn on top.
    """
    h = history_dict.get(model_name)
    if h is None:
        print(f"  [!] No history found for '{model_name}'"); return
 
    metrics = [
        ('train_loss',  'val_loss',  'Loss',     'lower left'),
        ('train_acc',   'val_acc',   'Accuracy', 'lower right'),
        ('val_auc',     None,        'Val AUC',  'lower right'),
    ]
 
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"Training Dynamics — {model_name}\n({FUSION_LABEL})",
        fontweight='bold'
    )
 
    for ax, (tr_key, vl_key, ylabel, loc) in zip(axes, metrics):
        epochs = np.arange(1, len(h[tr_key]) + 1)
 

        ax.scatter(epochs, h[tr_key], color=PALETTE['train'],
                   alpha=0.25, s=18, zorder=2)
 

        xs, ys = _smooth_curve(h[tr_key], kind='spline')
        ax.plot(xs + 1, ys, color=PALETTE['train'], label='Train', lw=2.4, zorder=3)
 
        if vl_key and vl_key in h:
            ax.scatter(epochs, h[vl_key], color=PALETTE['val'],
                       alpha=0.25, s=18, zorder=2)
            xs2, ys2 = _smooth_curve(h[vl_key], kind='spline')
            ax.plot(xs2 + 1, ys2, color=PALETTE['val'], label='Val',
                    lw=2.4, linestyle='--', zorder=3)
        else:

            xs2, ys2 = _smooth_curve(h[tr_key], kind='spline')
            ax.plot(xs2 + 1, ys2, color=PALETTE['apex'], lw=2.4)
 
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} vs. Epoch', fontweight='bold')
        ax.set_xlim([1, len(epochs)])
        if vl_key:
            ax.legend(loc=loc)
 
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
 
 
plot_training_curves(
    ALL_HISTORY,
    model_name='Apex_FusionTransformer',
    save_path=os.path.join(OUTPUT_DIR, 'R3_training_curves.png')
)

**CELL R-4  ·  Multiclass ROC Curve + Multiclass PR Curve (APEX)**

In [ ]:
 
def plot_roc_pr_multiclass(results_dict, model_name='Apex_FusionTransformer',
                           class_names=CLASS_NAMES, save_path=None):
    """
    One figure with two subplots:
      Left  → Multiclass ROC (one curve per class + macro-average)
      Right → Multiclass PR  (one curve per class + macro-average)
    Works for binary (2 classes) and multiclass (n > 2) equally.
    """
    res   = results_dict.get(model_name)
    if res is None:
        print(f"  [!] No results for '{model_name}'"); return
 
    y_true  = np.array(res['y_true'])
    y_prob  = np.array(res['y_prob'])      
    n_cls   = len(class_names)
 

    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(y_true, classes=np.arange(n_cls))
    if n_cls == 2:

        y_bin = np.hstack([1 - y_bin, y_bin])
    if y_prob.ndim == 1:
        probs = np.column_stack([1 - y_prob, y_prob])
    else:
        probs = y_prob
 
    colors_cls = [PALETTE['normal'], PALETTE['kc'],
                  '#8B5CF6', '#EF4444', '#0EA5E9'][:n_cls]
 
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        f"Multiclass ROC & PR Curves — {model_name}\n({FUSION_LABEL})",
        fontweight='bold'
    )
 

    fpr_all, tpr_all, roc_auc_all = {}, {}, {}
    for i, cname in enumerate(class_names):
        fp, tp, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc_all[i] = auc(fp, tp)
        fpr_all[i], tpr_all[i] = fp, tp
        xs, ys = _smooth_curve(tp, kind='spline')   
        ax_roc.plot(fp[np.linspace(0, len(fp)-1, len(ys)).astype(int)], ys,
                    color=colors_cls[i], lw=2.2,
                    label=f"{cname}  (AUC={roc_auc_all[i]:.3f})")
 

    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(n_cls)]))
    mean_tpr = np.mean([np.interp(all_fpr, fpr_all[i], tpr_all[i])
                        for i in range(n_cls)], axis=0)
    macro_auc = auc(all_fpr, mean_tpr)
    ax_roc.plot(all_fpr, mean_tpr, color='black', lw=2.5, linestyle='-.',
                label=f"Macro avg  (AUC={macro_auc:.3f})")
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Chance')
    ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title('Multiclass ROC Curve', fontweight='bold')
    ax_roc.legend(loc='lower right'); ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.02])
 

    pr_all, rec_all, ap_all = {}, {}, {}
    for i, cname in enumerate(class_names):
        prec, rec, _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ap_all[i]    = average_precision_score(y_bin[:, i], probs[:, i])
        pr_all[i], rec_all[i] = prec, rec
        idx = np.argsort(rec)
        xs_r = rec[idx]; ys_p = prec[idx]
        xs_s, ys_s = _smooth_curve(ys_p, kind='ma', window=7)
        ax_pr.plot(xs_r[np.linspace(0, len(xs_r)-1, len(ys_s)).astype(int)], ys_s,
                   color=colors_cls[i], lw=2.2,
                   label=f"{cname}  (AP={ap_all[i]:.3f})")
 

    macro_ap = np.mean(list(ap_all.values()))
    ax_pr.axhline(macro_ap, color='black', lw=2.5, linestyle='-.',
                  label=f"Macro AP = {macro_ap:.3f}")
    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_title('Multiclass Precision-Recall Curve', fontweight='bold')
    ax_pr.legend(loc='upper right'); ax_pr.set_xlim([0, 1]); ax_pr.set_ylim([0, 1.02])
 
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
    return {'macro_roc_auc': macro_auc, 'macro_ap': macro_ap}
 
 
_roc_pr_stats = plot_roc_pr_multiclass(
    ALL_RESULTS,
    model_name='Apex_FusionTransformer',
    class_names=CLASS_NAMES,
    save_path=os.path.join(OUTPUT_DIR, 'R4_roc_pr_multiclass.png')
)
 

**CELL R-5  ·  Confusion Matrix (labelled with Fusion Technique)**

In [ ]:
 
def plot_labelled_confusion_matrix(results_dict, model_name='Apex_FusionTransformer',
                                   class_names=CLASS_NAMES, save_path=None):
    """
    Confusion matrix with absolute counts AND row-normalised percentages.
    Subtitle shows the exact fusion strategy used in the APEX model.
    """
    res    = results_dict.get(model_name)
    if res is None:
        print(f"  [!] No results for '{model_name}'"); return
 
    y_true = np.array(res['y_true'])
    y_pred = np.array(res['y_pred'])
    cm     = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
 
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm_pct, interpolation='nearest', cmap='Blues',
                   vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Row-Normalised (%)', rotation=270, labelpad=16)
 
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks); ax.set_xticklabels(class_names, rotation=30, ha='right')
    ax.set_yticks(ticks); ax.set_yticklabels(class_names)
 
    thresh = cm_pct.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm_pct[i, j] > thresh else 'black'
            ax.text(j, i, f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)",
                    ha='center', va='center', color=color,
                    fontsize=12, fontweight='bold')
 
    ax.set_ylabel('True Label', fontweight='bold')
    ax.set_xlabel('Predicted Label', fontweight='bold')
    ax.set_title(
        f"Confusion Matrix — {model_name}\n"
        f"Fusion: Bidirectional Cross-Attention Intermediate Fusion",
        fontweight='bold', pad=14
    )
 
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show(); plt.close()
 
 
plot_labelled_confusion_matrix(
    ALL_RESULTS,
    model_name='Apex_FusionTransformer',
    class_names=CLASS_NAMES,
    save_path=os.path.join(OUTPUT_DIR, 'R5_confusion_matrix.png')
)

**CELL R-6  ·  Cross-Validation — Summary Table + Comparison Bar Plot**

In [ ]:
# 4: 5-Fold Stratified Patient-Level Cross-Validation
# ──────────────────────────────────────────────────────────────
# Replaces (or supplements) the single 70/15/15 holdout with robust
# 5-fold CV where folds are split on UNIQUE PATIENT IDs, guaranteeing
# zero bilateral leakage across folds.
#
# Each fold:
#   1. Splits unique patients stratified by patient-level label
#   2. Fits ClinicalPreprocessor on fold-train ONLY (no leakage)
#   3. Fits per-channel registrars on fold-train
#   4. Trains model for n_epochs with early stopping
#   5. Evaluates on fold-val; bootstraps AUC / F1 / MCC CIs (N=2000)

from sklearn.model_selection import StratifiedGroupKFold

def _bootstrap_ci(y_true, y_prob, y_pred, metric_fn, n_boot=N_BOOTSTRAP, seed=SEED):
    """
    Compute 95% bootstrap CI for a scalar metric.
    metric_fn(y_true, y_pred_or_prob) -> float
    """
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = []
    for _ in range(n_boot):
        bi = rng.choice(idx, size=len(idx), replace=True)
        # guard: need both classes in bootstrap sample
        if len(np.unique(y_true[bi])) < 2:
            continue
        try:
            vals.append(metric_fn(y_true[bi], y_prob[bi] if y_prob is not None else y_pred[bi]))
        except Exception:
            pass
    if len(vals) < 10:
        return float('nan'), float('nan')
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def run_stratified_cv(
    model_class,
    model_kwargs,
    full_df,                     # combined DataFrame (patient_code, label, image paths)
    full_clin_array,             # numpy array: clinical features BEFORE preprocessing
    full_clin_df,                # raw clinical DataFrame aligned to full_df (for re-fitting)
    registrars_pretrained=None,  # optional pre-fitted registrars (will be re-fitted per fold)
    n_folds   = 5,
    n_epochs  = NUM_EPOCHS,
    patience  = PATIENCE,
    device    = DEVICE,
    save_path_csv  = None,
    save_path_plot = None,
):
    """
    5-Fold Stratified Patient-Level Cross-Validation.

    Parameters
    ----------
    model_class      : nn.Module subclass (e.g. AttentionIntermediateFusionNet)
    model_kwargs     : dict of kwargs passed to model_class(**model_kwargs) per fold
    full_df          : pd.DataFrame with columns patient_code, label, and image paths
    full_clin_array  : ignored (kept for API compat); preprocessing is re-done per fold
    full_clin_df     : raw pd.DataFrame with clinical columns aligned to full_df
    n_folds          : number of CV folds (default 5)
    n_epochs         : max epochs per fold
    patience         : early stopping patience per fold
    """
    from sklearn.model_selection import StratifiedGroupKFold

    assert 'patient_code' in full_df.columns, "full_df must contain 'patient_code'"
    assert 'label' in full_df.columns, "full_df must contain 'label'"

    metric_keys = ['accuracy', 'balanced_accuracy', 'auc_roc', 'auc_pr',
                   'sensitivity', 'specificity', 'f1_score', 'mcc']

    # ── Patient-level group construction ──────────────────────────
    groups = full_df['patient_code'].values          # one group per eye-row
    y_rows = full_df['label'].values                 # row-level labels

    # StratifiedGroupKFold ensures:
    #   (a) each patient's eyes always land in ONE fold
    #   (b) class balance is approximately maintained across folds
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    rows_cv = []
    print(f"\n{'─'*62}")
    print(f"  5-Fold Stratified Patient-Level Cross-Validation")
    print(f"  Model   : {model_class.__name__}")
    print(f"  Epochs  : {n_epochs} per fold  |  Patience: {patience}")
    print(f"  Folds   : {n_folds}  |  Bootstrap CI samples: {N_BOOTSTRAP}")
    print(f"  Unique patients in pool: {full_df['patient_code'].nunique()}")
    print(f"{'─'*62}")

    all_idx = np.arange(len(full_df))

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(all_idx, y_rows, groups)):
        fold_num = fold + 1
        tr_df = full_df.iloc[tr_idx].reset_index(drop=True)
        vl_df = full_df.iloc[vl_idx].reset_index(drop=True)
        tr_raw_clin = full_clin_df.iloc[tr_idx].reset_index(drop=True)
        vl_raw_clin = full_clin_df.iloc[vl_idx].reset_index(drop=True)

        # Verify zero patient overlap
        tr_pts = set(tr_df['patient_code'].unique())
        vl_pts = set(vl_df['patient_code'].unique())
        assert len(tr_pts & vl_pts) == 0, f"Fold {fold_num}: patient overlap detected!"

        print(f"\n  [Fold {fold_num}/{n_folds}]")
        print(f"    Train: {len(tr_df):4d} eye-rows | {len(tr_pts):4d} patients "
              f"| KC={tr_df['label'].sum()}")
        print(f"    Val  : {len(vl_df):4d} eye-rows | {len(vl_pts):4d} patients "
              f"| KC={vl_df['label'].sum()}")

        # ── Step A: Fit clinical preprocessor on fold-train ONLY ──────
        fold_clin_pp = ClinicalPreprocessor(imputer='knn', scaler='minmax',
                                             cat_encoding='onehot', dim_reduction='none')
        clin_tr = fold_clin_pp.fit_transform(tr_df)
        clin_vl = fold_clin_pp.transform(vl_df)
        fold_cdim = clin_tr.shape[1]

        # ── Step B: Fit per-channel registrars on fold-train ──────────
        fold_regs = fit_per_channel_registrars(tr_df, n_samples=min(200, len(tr_df)))

        # ── Step C: Normalizer stats on fold-train ────────────────────
        stats_ds_fold = OrbscanDataset(
            tr_df, clin_tr,
            transform=get_val_transforms(), augment=False,
            registrar=fold_regs,
            normalizer_means=None, normalizer_stds=None
        )
        fold_means, fold_stds = build_normalizer(stats_ds_fold, n_samples=min(500, len(tr_df)))
        del stats_ds_fold

        # ── Step D: Build dataloaders ──────────────────────────────────
        tr_ds = OrbscanDataset(tr_df, clin_tr,
                                transform=get_train_transforms(), augment=True,
                                registrar=fold_regs,
                                normalizer_means=fold_means, normalizer_stds=fold_stds)
        vl_ds = OrbscanDataset(vl_df, clin_vl,
                                transform=get_val_transforms(), augment=False,
                                registrar=fold_regs,
                                normalizer_means=fold_means, normalizer_stds=fold_stds)

        # Weighted sampler for class imbalance in training fold
        tr_labels = tr_df['label'].values
        cls_counts = np.bincount(tr_labels)
        cls_weights = 1.0 / (cls_counts + 1e-8)
        sample_wts  = torch.tensor([cls_weights[l] for l in tr_labels], dtype=torch.float32)
        sampler = WeightedRandomSampler(sample_wts, num_samples=len(sample_wts), replacement=True)

        tr_ldr = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
        vl_ldr = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # ── Step E: Instantiate & train model ─────────────────────────
        fold_kwargs = {**model_kwargs, 'clinical_dim': fold_cdim}
        m = model_class(**fold_kwargs).to(device)
        m, _ = train_model(m, tr_ldr, vl_ldr,
                            model_name=f'CV_Fold{fold_num}',
                            n_epochs=n_epochs,
                            patience=patience,
                            device=device)

        # ── Step F: Evaluate & collect metrics ────────────────────────
        _, _, yt, yp, ypr = evaluate(m, vl_ldr, FocalLoss(), device)
        met = compute_all_metrics(yt, yp, ypr)

        # ── Step G: Bootstrap 95% CIs for AUC, F1, MCC ───────────────
        from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef

        auc_lo, auc_hi = _bootstrap_ci(
            yt, ypr, None,
            lambda yt_b, ypr_b: roc_auc_score(yt_b, ypr_b),
            n_boot=N_BOOTSTRAP
        )
        f1_lo, f1_hi = _bootstrap_ci(
            yt, None, yp,
            lambda yt_b, yp_b: f1_score(yt_b, yp_b, zero_division=0),
            n_boot=N_BOOTSTRAP
        )
        mcc_lo, mcc_hi = _bootstrap_ci(
            yt, None, yp,
            lambda yt_b, yp_b: matthews_corrcoef(yt_b, yp_b),
            n_boot=N_BOOTSTRAP
        )

        met['fold']           = fold_num
        met['n_val_patients'] = len(vl_pts)
        met['auc_ci_lo']      = auc_lo
        met['auc_ci_hi']      = auc_hi
        met['f1_ci_lo']       = f1_lo
        met['f1_ci_hi']       = f1_hi
        met['mcc_ci_lo']      = mcc_lo
        met['mcc_ci_hi']      = mcc_hi

        rows_cv.append(met)
        print(f"  Fold {fold_num} → AUC={met['auc_roc']:.4f} "
              f"[{auc_lo:.3f}–{auc_hi:.3f}]  "
              f"F1={met['f1_score']:.4f} [{f1_lo:.3f}–{f1_hi:.3f}]  "
              f"MCC={met['mcc']:.4f} [{mcc_lo:.3f}–{mcc_hi:.3f}]")

    # ── Aggregate results ──────────────────────────────────────────
    cv_df = pd.DataFrame(rows_cv)
    front_cols = ['fold', 'n_val_patients'] + metric_keys +                  ['auc_ci_lo', 'auc_ci_hi', 'f1_ci_lo', 'f1_ci_hi',
                  'mcc_ci_lo', 'mcc_ci_hi']
    cv_df = cv_df[[c for c in front_cols if c in cv_df.columns]]

    summary = cv_df[metric_keys].agg(['mean', 'std']).T.reset_index()
    summary.columns = ['metric', 'mean', 'std']
    summary['CI_95_mean_pm_std'] = summary.apply(
        lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)

    print(f"\n{'═'*62}")
    print(f"  {n_folds}-Fold CV Summary  —  {model_class.__name__}")
    print(f"{'═'*62}")
    print(summary[['metric', 'CI_95_mean_pm_std']].to_string(index=False))
    print(f"{'═'*62}")

    if save_path_csv:
        cv_df.to_csv(save_path_csv, index=False)
        summary.to_csv(save_path_csv.replace('.csv', '_summary.csv'), index=False)
        print(f"  Saved → {save_path_csv}")

    if save_path_plot:
        PALETTE_CV = {'apex': '#2563EB'}
        fig, ax = plt.subplots(figsize=(14, 5.5))
        x = np.arange(len(metric_keys))
        means = cv_df[metric_keys].mean().values
        stds  = cv_df[metric_keys].std().values
        bars  = ax.bar(x, means, yerr=stds, capsize=5,
                       color=PALETTE_CV['apex'], alpha=0.82,
                       ecolor='#374151', error_kw={'linewidth': 1.6})
        for fi, frow in cv_df.iterrows():
            ax.scatter(x, frow[metric_keys].values,
                       color='black', s=22, alpha=0.55, zorder=3)
        ax.set_xticks(x)
        ax.set_xticklabels([k.replace('_', '\n') for k in metric_keys], fontsize=10)
        ax.set_ylim([0, 1.15])
        ax.set_ylabel('Score')
        ax.set_title(
            f"{n_folds}-Fold Stratified Patient-Level CV — {model_class.__name__}\n"
            f"Bars = mean ± std  |  Dots = per-fold values  |  "
            f"Bootstrap CIs N={N_BOOTSTRAP}",
            fontweight='bold'
        )
        for bar, m, s in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width() / 2.,
                    bar.get_height() + s + 0.01,
                    f"{m:.3f}", ha='center', va='bottom', fontsize=9)
        fig.savefig(save_path_plot, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path_plot}")
        plt.show(); plt.close()

    return cv_df, summary


# ── Execution block (un-comment after verifying full_df and full_clin_df) ──
# This combines all labelled eye-rows for patient-level grouped CV.

# _full_combined_df  = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
# _full_combined_raw = pd.concat([               # raw clinical (before any transform)
#     train_df, val_df, test_df                  # DataAuditor already has merged clinical cols
# ]).reset_index(drop=True)

# cv_results_df, cv_summary_df = run_stratified_cv(
#     model_class       = AttentionIntermediateFusionNet,
#     model_kwargs      = {},                    # clinical_dim injected per fold automatically
#     full_df           = _full_combined_df,
#     full_clin_array   = None,                  # unused — re-fitted per fold
#     full_clin_df      = _full_combined_raw,    # raw clinical rows for ClinicalPreprocessor
#     n_folds           = 5,
#     n_epochs          = NUM_EPOCHS,
#     patience          = PATIENCE,
#     save_path_csv     = os.path.join(OUTPUT_DIR, 'R6_cv_results.csv'),
#     save_path_plot    = os.path.join(OUTPUT_DIR, 'R6_cv_bar_plot.png'),
# )


**CELL R-7  ·  Grad-CAM Re-Assessment (Fixed for ViT)**

In [ ]:
class GradCAMViTFixed:

 
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device).eval()
        self.device = device
 

    def _get_cam(self, img_t, clin_t, target_class=1):
        """
        img_t  : (1, C, H, W)  – single image batch on CPU
        clin_t : (1, D)        – single clinical batch on CPU
        Returns heatmap: np.ndarray (H, W) in [0, 1]
        """
        self.model.train()
        for m in self.model.modules():
            if isinstance(m, (nn.Dropout, nn.Dropout2d)):
                m.eval()
 
        enc = self.model.img_encoder
 
        if _TIMM_OK and hasattr(enc, 'vit') and hasattr(enc.vit, 'blocks'):
            last_block = enc.vit.blocks[-1]
        elif hasattr(enc, '_fb') and hasattr(enc._fb, 'tr'):
            last_block = enc._fb.tr.layers[-1]  
        else:
            last_block = list(enc.modules())[-1]
 
        _feat     = [None]
        _grad     = [None]
 
        def _fwd_hook(module, inp, out):
            _feat[0] = out         
 
        def _bwd_hook(module, grad_in, grad_out):
            _grad[0] = grad_out[0].detach()  
 
        h_f = last_block.register_forward_hook(_fwd_hook)
        h_b = last_block.register_full_backward_hook(_bwd_hook)
  
        img_d  = img_t.to(self.device)
        clin_d = clin_t.to(self.device)
 
        self.model.zero_grad()
        with torch.enable_grad():
            logits = self.model(img_d, clin_d)
            score  = logits[0, target_class]
            score.backward()
 
        h_f.remove()
        h_b.remove()
 
        if _feat[0] is None or _grad[0] is None:
            print("  [GradCAM] WARNING: hooks returned None — returning blank map.")
            return np.zeros((IMG_SIZE, IMG_SIZE))
 
        feat = _feat[0].detach()            # (1, N+1, D)  N = num_patches, +1 = CLS
        grad = _grad[0]                     # (1, N+1, D)
 
        feat_p = feat[0, 1:]                # (N, D)
        grad_p = grad[0, 1:]                # (N, D)
 
        alpha  = grad_p.mean(dim=-1)        # (N,)
 
        cam    = F.relu(alpha.unsqueeze(-1) * feat_p).sum(dim=-1)  # (N,)
        cam    = cam.cpu().numpy()
 
        n_patches = cam.shape[0]
        grid_side = int(math.sqrt(n_patches))
        cam_grid  = cam[: grid_side * grid_side].reshape(grid_side, grid_side)

        c_min, c_max = cam_grid.min(), cam_grid.max()
        if c_max > c_min:
            cam_grid = (cam_grid - c_min) / (c_max - c_min)
        else:
            cam_grid = np.zeros_like(cam_grid)
 
        heatmap = np.array(
            Image.fromarray((cam_grid * 255).astype(np.uint8))
                 .resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
        ) / 255.0
 
        self.model.eval()
        return heatmap
 
    def visualize_batch(self, img_batch, clin_batch, labels=None,
                        n_samples=5, save_path=None,
                        target_class=1):
        """
        img_batch  : (B, C, H, W)  on CPU
        clin_batch : (B, D)        on CPU
        labels     : (B,) optional – shown in subtitle
        """
        N    = min(n_samples, img_batch.size(0))
        cols = NUM_CHANNELS + 1   
        fig  = plt.figure(figsize=(4.8 * cols, 4.2 * N))
        fig.suptitle(
            f"Grad-CAM Re-Assessment — {type(self.model).__name__}\n"
            f"Fusion: Bidirectional Cross-Attention  |  "
            f"Target class: {CLASS_NAMES[target_class]}\n"
            f"Heatmap: 'inferno' colormap overlaid on each Orbscan modality",
            fontweight='bold', y=1.01
        )
        gs = gridspec.GridSpec(N, cols, figure=fig,
                               hspace=0.35, wspace=0.15)
 
        cmap_overlay = matplotlib.colormaps['inferno']
 
        for i in range(N):
            img_i  = img_batch[i:i+1]
            clin_i = clin_batch[i:i+1]
            lbl    = int(labels[i]) if labels is not None else target_class
 
            heatmap = self._get_cam(img_i, clin_i, target_class=lbl)
 
            img_np = img_i[0].cpu().numpy()   # (C, H, W)
 
            ax0 = fig.add_subplot(gs[i, 0])
            ax0.imshow(heatmap, cmap='inferno', vmin=0, vmax=1, aspect='auto')
            ax0.set_title(f"Heatmap\n(sample {i+1})", fontsize=10, fontweight='bold')
            ax0.axis('off')
            if labels is not None:
                ax0.set_xlabel(f"True: {CLASS_NAMES[lbl]}", fontsize=9)
 
            for j, mt in enumerate(MAP_TYPES):
                ax = fig.add_subplot(gs[i, j + 1])
                mod_img = img_np[j]                    # (H, W) normalised
 
                m_min, m_max = mod_img.min(), mod_img.max()
                if m_max > m_min:
                    mod_disp = (mod_img - m_min) / (m_max - m_min)
                else:
                    mod_disp = np.zeros_like(mod_img)
 
                base_rgb = np.stack([mod_disp] * 3, axis=-1)
 
                heat_rgba = cmap_overlay(heatmap)
                alpha_v   = 0.55
                blended   = (1 - alpha_v) * base_rgb + alpha_v * heat_rgba[..., :3]
                blended   = np.clip(blended, 0, 1)
 
                ax.imshow(blended, aspect='auto')
                ax.set_title(mt, fontsize=10, fontweight='bold')
                ax.axis('off')
 
                if j == len(MAP_TYPES) - 1:
                    sm = plt.cm.ScalarMappable(cmap='inferno',
                                               norm=plt.Normalize(0, 1))
                    sm.set_array([])
                    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
                    cbar.set_label('Activation', fontsize=8, rotation=270, labelpad=12)
                    cbar.ax.tick_params(labelsize=7)
 
        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  Saved → {save_path}")
        plt.show(); plt.close()
 
 
print("\n   Grad-CAM Re-Assessment on APEX model ")
print("  Targeting: last ViT transformer block (blocks[-1])")
print("  Grad fix : model.train() + Dropout.eval() + enable_grad()")
 
gradcam_fixed = GradCAMViTFixed(apex_model, DEVICE)
 
_gcam_imgs, _gcam_clin, _gcam_labels = next(iter(TEL))
 
gradcam_fixed.visualize_batch(
    _gcam_imgs, _gcam_clin,
    labels    = _gcam_labels.numpy(),
    n_samples = 5,
    target_class = 1,      # Keratoconus
    save_path = os.path.join(OUTPUT_DIR, 'R7_gradcam_fixed.png')
)

**CELL R-8  ·  Blueprint CSV Export**

In [ ]:


_apex_res = ALL_RESULTS.get('Apex_FusionTransformer', {})
if _apex_res:
    _apex_metrics = compute_all_metrics(
        _apex_res['y_true'], _apex_res['y_pred'], _apex_res['y_prob']
    )
    cv_summary_df = pd.DataFrame([
        {'metric': k, 'mean': v, 'std': float('nan')}
        for k, v in _apex_metrics.items()
    ])
else:
    cv_summary_df = pd.DataFrame(columns=['metric', 'mean', 'std'])

print("  [R-8] cv_summary_df built from test-set metrics (CV skipped).")


def export_blueprint_csv(results_dict, cv_summary_df,
                         model_name='Apex_FusionTransformer',
                         save_path=None):
    rows = []

    for mn, res in results_dict.items():
        yt, yp, ypr = res['y_true'], res['y_pred'], res['y_prob']
        m = compute_all_metrics(yt, yp, ypr)

        if mn == 'Apex_FusionTransformer':
            fstrat = 'Bidirectional Cross-Attention Intermediate Fusion'
        elif mn.startswith('Fusion_early'):
            fstrat = 'Early Fusion (Concat)'
        elif mn.startswith('Fusion_late'):
            fstrat = 'Late Fusion (Average / Voting / Meta)'
        elif mn.startswith('Fusion_intermediate'):
            fstrat = 'Intermediate Fusion (MLP Concat)'
        elif mn.startswith('Fusion_hybrid'):
            fstrat = 'Hybrid Fusion (Early+Cross-Attn+Late)'
        elif mn.startswith('Ablation'):
            fstrat = f'Ablation — {mn.replace("Ablation_", "")}'
        elif any(x in mn for x in ['DenseNet', 'ResNet', 'EfficientNet']):
            fstrat = 'Unimodal Baseline (Image Only)'
        elif 'TabOnly' in mn:
            fstrat = 'Unimodal Baseline (Clinical Only)'
        else:
            fstrat = 'Baseline'

        row = {
            'Model'             : mn,
            'Fusion_Strategy'   : fstrat,
            'Accuracy'          : round(m['accuracy'],          4),
            'Balanced_Accuracy' : round(m['balanced_accuracy'], 4),
            'AUC_ROC'           : round(m['auc_roc'],           4),
            'AUC_PR'            : round(m['auc_pr'],            4),
            'Sensitivity'       : round(m['sensitivity'],       4),
            'Specificity'       : round(m['specificity'],       4),
            'F1_Score'          : round(m['f1_score'],          4),
            'MCC'               : round(m['mcc'],               4),
        }

        if mn == model_name:
            def _get(metric_name, col):
                row_ = cv_summary_df.loc[cv_summary_df['metric'] == metric_name]
                val  = row_[col].values[0] if len(row_) else float('nan')
                return round(float(val), 4) if not (isinstance(val, float) and math.isnan(val)) else 'N/A'
            row['CV_AUC_mean'] = _get('auc_roc',  'mean')
            row['CV_AUC_std']  = _get('auc_roc',  'std')
            row['CV_F1_mean']  = _get('f1_score', 'mean')
            row['CV_F1_std']   = _get('f1_score', 'std')
            row['CV_Note']     = 'Test-set estimate (K-Fold skipped)'
        else:
            row.update({'CV_AUC_mean': 'N/A', 'CV_AUC_std': 'N/A',
                        'CV_F1_mean': 'N/A',  'CV_F1_std':  'N/A',
                        'CV_Note':    'N/A'})

        rows.append(row)

    blueprint_df = pd.DataFrame(rows)
    blueprint_df['_is_apex'] = (blueprint_df['Model'] == model_name).astype(int)
    blueprint_df = (blueprint_df
                    .sort_values(['_is_apex', 'AUC_ROC'], ascending=[False, False])
                    .drop(columns=['_is_apex'])
                    .reset_index(drop=True))

    print("\n  Blueprint Document CSV — All Models")
    print(blueprint_df[['Model', 'Fusion_Strategy', 'AUC_ROC', 'F1_Score',
                         'Sensitivity', 'Specificity', 'CV_AUC_mean']].to_string(index=False))

    if save_path:
        blueprint_df.to_csv(save_path, index=False)
        print(f"\n  Saved → {save_path}")

    return blueprint_df


blueprint_df = export_blueprint_csv(
    results_dict  = ALL_RESULTS,
    cv_summary_df = cv_summary_df,
    model_name    = 'Apex_FusionTransformer',
    save_path     = os.path.join(OUTPUT_DIR, 'R8_blueprint_results.csv')
)


print("  RESULTS AND EVALUATION SUITE COMPLETE")
print(f"  All artefacts saved to : {OUTPUT_DIR}")

print("  Files produced:")
for fn in ['R1_raw_samples.png', 'R2_preprocessed_samples.png',
           'R2_class_distribution.png', 'R3_training_curves.png',
           'R4_roc_pr_multiclass.png', 'R5_confusion_matrix.png',
           'R7_gradcam_fixed.png',     'R8_blueprint_results.csv']:
    full   = os.path.join(OUTPUT_DIR, fn)
    exists = '✓' if os.path.exists(full) else '✗'
    print(f"    {exists}  {fn}")

**CELL R-9  ·  Smooth Accuracy & Loss Curves — Proposed Model (APEX)**

In [ ]:




from scipy.interpolate import make_interp_spline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family'      : 'serif',
    'font.size'        : 12,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 12,
    'legend.fontsize'  : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.30,
    'grid.linestyle'   : '--',
    'lines.linewidth'  : 2.2,
    'figure.dpi'       : 130,
})

def _bspline_smooth(y, n_out=400):
    """
    Fit a cubic B-spline through a reduced set of control points.
    Returns (x_smooth, y_smooth) for plotting.
    Automatically falls back to moving-average for very short histories.
    """
    x = np.arange(len(y), dtype=float)
    if len(y) < 6:
        kernel = np.ones(3) / 3
        return x, np.convolve(y, kernel, mode='same')
    n_ctrl  = max(4, len(y) // 3)
    x_ctrl  = np.linspace(0, len(y) - 1, n_ctrl)
    y_ctrl  = np.interp(x_ctrl, x, y)
    spline  = make_interp_spline(x_ctrl, y_ctrl, k=3)
    x_new   = np.linspace(0, len(y) - 1, n_out)
    return x_new, spline(x_new)


MODEL_NAME = 'Apex_FusionTransformer'
h = ALL_HISTORY.get(MODEL_NAME)

if h is None:
    print(f"[!] No training history found for '{MODEL_NAME}'. "
          "Make sure Cell 15 (APEX training) has been executed.")
else:
    epochs     = np.arange(1, len(h['train_acc']) + 1)
    COLOR_TRAIN = '#2563EB'  
    COLOR_VAL   = '#F97316'  

    fig, (ax_acc, ax_loss) = plt.subplots(
        1, 2, figsize=(14, 5.2),
        gridspec_kw={'wspace': 0.32}
    )

    fig.suptitle(
        f"Learning Curves — Proposed Model  ({MODEL_NAME})\n"
        "Bidirectional Cross-Attention Intermediate Fusion  (ViT-B/16 + TabTransformer)",
        fontsize=13, fontweight='bold', y=1.02
    )

    ax_acc.scatter(epochs, h['train_acc'], color=COLOR_TRAIN, alpha=0.18, s=14)
    ax_acc.scatter(epochs, h['val_acc'],   color=COLOR_VAL,   alpha=0.18, s=14)

    xs, ys = _bspline_smooth(h['train_acc'])
    ax_acc.plot(xs + 1, ys, color=COLOR_TRAIN, label='Train Acc', lw=2.3)

    xs2, ys2 = _bspline_smooth(h['val_acc'])
    ax_acc.plot(xs2 + 1, ys2, color=COLOR_VAL,  label='Val Acc',   lw=2.3)

    ax_acc.set_xlabel('Epochs'); ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title('Training vs. Validation Accuracy', fontweight='bold')
    ax_acc.set_xlim([1, len(epochs)])
    ax_acc.legend(loc='lower right', framealpha=0.85)

    ax_loss.scatter(epochs, h['train_loss'], color=COLOR_TRAIN, alpha=0.18, s=14)
    ax_loss.scatter(epochs, h['val_loss'],   color=COLOR_VAL,   alpha=0.18, s=14)

    xs3, ys3 = _bspline_smooth(h['train_loss'])
    ax_loss.plot(xs3 + 1, ys3, color=COLOR_TRAIN, label='Train Loss', lw=2.3)

    xs4, ys4 = _bspline_smooth(h['val_loss'])
    ax_loss.plot(xs4 + 1, ys4, color=COLOR_VAL,   label='Val Loss',   lw=2.3)

    ax_loss.set_xlabel('Epochs'); ax_loss.set_ylabel('Loss')
    ax_loss.set_title('Training vs. Validation Loss', fontweight='bold')
    ax_loss.set_xlim([1, len(epochs)])
    ax_loss.legend(loc='upper right', framealpha=0.85)

    _save = os.path.join(OUTPUT_DIR, 'R3b_acc_loss_curves_apex.png')
    fig.savefig(_save, dpi=150, bbox_inches='tight')
    print(f"  Saved → {_save}")
    plt.show(); plt.close()

In [ ]:

print(f"  N_BOOTSTRAP used        : {N_BOOTSTRAP}")
print(f"  Primary CNN backbone    : DenseNet-121  (plan-aligned)")
print(f"  Image encoder (Apex)    : PretrainedViTEncoder (timm ViT-B/16)")
print(f"  Clinical encoder (Apex) : TabTransformerEncoder (CLS-token pooling)")
print(f"  Models trained          : {len(ALL_RESULTS)}")
print(f"  Output directory        : {OUTPUT_DIR}")
print("\n  Notebook-- COMPLETE.")
